# Visualizing Relationship Between μ Structure and Slip Inversion

This notebook visualizes:
1. μ structure on fault surface (from 3D model)
2. Slip comparison: 3D inversion vs. homogeneous inversion
3. Displacement difference at GNSS stations
4. μ contrast across fault

**Case:** nicoyaCK4, 2-stripe pattern, DeShon3D_ref_4 vs mul40u40

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
import pyvista as pv
import pygmt

# FEniCS for mesh loading
import dolfin as dl

print("Libraries imported successfully!")

## 1. Setup and Configuration

In [ ]:
# Paths
datadir = "/home/staff/chao/SSEinv/Nicoya/data/"
meshpath = "/home/staff/chao/SSEinv/Nicoya/mesh/"
resultpath = "/home/staff/chao/SSEinv/Nicoya/syn_slip/"

PLATETYPE = "CK"
# PLATETYPE = "SLAB2"

# Mesh and slip pattern
### fault surface with rough local topography 
# meshname = "nicoyaCK4"  
meshname = "nicoyaCKden_sm"   # based on nicoyaCK3 or 4, but denser mesh size, and smaller fault zone

flag_savefig = 0

slip_str_gt = "_stripe_x0_y-45_lx80_dx35_rot0_ms0.0785_pou"

# Structure strings
het3d_str = "_DeShon3D_ref_4"  # 3D heterogeneous model, amplified, CG_mu=1
# het3d_str = "_DeShon3D_ref_1"  # 3D heterogeneous model, original, CG_mu=1

# het3d_str = "_DeShon3D_ref_4_CGmu2"  # 3D heterogeneous model, amplified, using order 2 CG for mu

het3d_str = '_DeShon3D_ref_4_hull'  # 3D heterogeneous model, amplified, CG_mu=1, convex hull version

# het3d_str = "_mul55u25"

homo_str = "_mul40u40"          # Homogeneous reference

# Inversion string
# inv_str = "_synlockbd_w945390_gs1e+02_ds1e-07"

inv_str = "_synlockbd_w945390_gs2.5e+02_ds2.5e-07"

# Fault boundary ID
fault_id = 7

# Subdomain IDs for Δμ calculation
blockleft = 8   # subducting plate (footwall)
blockright = 9  # overriding plate (hanging wall)

# Problem type for slip loading
# For locking problems: slip files are in meters, normalize by amp to get coupling ratio
problem_type = 'locking'  # 'coseismic' or 'locking'
V_norm = 78.5 / 1e3  # Trench-normal long-term loading: 78.5 mm/yr = 0.0785 m/yr
amp = V_norm  # Normalization amplitude (for locking: max coupling = 1 = complete locking)

print(f"Configuration:")
print(f"  Mesh: {meshname}")
print(f"  Slip pattern: {slip_str_gt}")
print(f"  3D structure: {het3d_str}")
print(f"  Homogeneous: {homo_str}")
print(f"  Problem type: {problem_type}")
print(f"  Normalization amp: {amp:.4f} m/yr (V_norm = {V_norm*1e3:.1f} mm/yr)")

## 2. Load Mesh and Extract Fault Facets

**Note:** We use fault **facets** (triangular faces) rather than vertices throughout this analysis because:
- Normals are naturally defined on facets
- μ contrast calculation requires facet centers and normals
- Provides ~2× more sampling points than vertices (~6185 facets vs ~3165 vertices)

In [ ]:
# Load mesh
mesh = dl.Mesh(meshpath + meshname + '.xml')
boundaries = dl.MeshFunction("size_t", mesh, meshpath + meshname + '_facet_region.xml')
subdomains = dl.MeshFunction("size_t", mesh, meshpath + meshname + '_physical_region.xml')

print(f"Mesh loaded: {mesh.num_vertices()} vertices, {mesh.num_cells()} cells")
print(f"Mesh dimension: {mesh.topology().dim()}")

In [ ]:
# Load fault geometry (same coordinates used for slip data)
fault_geom_file = resultpath + f"fault_geometry_{meshname}.txt"
fault_geom = pd.read_csv(fault_geom_file, sep=r'\s+', skiprows=lambda x: x % 3 != 1,
                        names=['x', 'y', 'z'])
fault_coords = fault_geom[['x', 'y', 'z']].values

print(f"Loaded fault coordinates: {len(fault_coords)} nodes (matching slip data)")

# Compute normals at fault vertices by averaging nearby facet normals
def compute_vertex_normals(mesh, boundaries, fault_id, vertex_coords):
    """Compute normals at fault vertices by averaging surrounding facet normals"""
    from scipy.spatial import cKDTree

    # First get all facet centers and normals
    facet_centers = []
    facet_normals_list = []

    for facet in dl.facets(mesh):
        if boundaries[facet] == fault_id:
            vertices = facet.entities(0)
            coords = mesh.coordinates()[vertices]
            center = coords.mean(axis=0)

            v1 = coords[1] - coords[0]
            v2 = coords[2] - coords[0]
            normal = np.cross(v1, v2)
            normal = normal / np.linalg.norm(normal)

            facet_centers.append(center)
            facet_normals_list.append(normal)

    facet_centers = np.array(facet_centers)
    facet_normals_array = np.array(facet_normals_list)

    # Build KDTree for nearest neighbor lookup
    tree = cKDTree(facet_centers)

    # For each vertex, average normals from 5 nearest facets
    vertex_normals = []
    for vc in vertex_coords:
        _, indices = tree.query(vc, k=5)
        avg_normal = facet_normals_array[indices].mean(axis=0)
        avg_normal = avg_normal / np.linalg.norm(avg_normal)
        vertex_normals.append(avg_normal)

    return np.array(vertex_normals)

print("Computing normals at fault vertices...")

fault_normals = compute_vertex_normals(mesh, boundaries, fault_id, fault_coords)

# Determine and correct normal direction
def determine_normal_direction(mesh, subdomains, coords, normals, blockleft_id, blockright_id):
    """Ensure normals point from FW (blockleft) to HW (blockright)"""
    n_test = min(10, len(coords))
    flip_count = 0

    for i in range(n_test):
        center = coords[i]
        normal = normals[i]
        offset_test = 100.0
        point_plus = center + offset_test * normal
        point_minus = center - offset_test * normal

        cell_plus = mesh.bounding_box_tree().compute_first_entity_collision(dl.Point(point_plus))
        cell_minus = mesh.bounding_box_tree().compute_first_entity_collision(dl.Point(point_minus))

        if cell_plus < mesh.num_cells() and cell_minus < mesh.num_cells():
            subdomain_plus = subdomains[int(cell_plus)]
            subdomain_minus = subdomains[int(cell_minus)]

            # Normal should point from FW (blockleft) to HW (blockright)
            # So: normal direction should have blockright, -normal should have blockleft
            if subdomain_minus == blockright_id and subdomain_plus == blockleft_id:
                flip_count += 1

    should_flip = flip_count > n_test // 2

    if should_flip:
        print(f"Flipping normal direction to point from FW to HW")
        return -normals
    else:
        print(f"Normal direction already correct (FW→HW)")
        return normals.copy()

fault_normals = determine_normal_direction(mesh, subdomains, fault_coords, fault_normals,
blockleft, blockright)


def spatial_median_filter(coords, values, k_neighbors=10):
    """Apply spatial median filter using k nearest neighbors"""
    from scipy.spatial import cKDTree
    
    tree = cKDTree(coords)
    filtered = np.zeros_like(values)

    for i in range(len(values)):
        if np.isnan(values[i]):
            filtered[i] = np.nan
            continue

        # Find k nearest neighbors
        distances, indices = tree.query(coords[i], k=k_neighbors+1)  # +1 includes self
        neighbor_values = values[indices[1:]]  # Exclude self

        # Remove NaN neighbors
        neighbor_values = neighbor_values[~np.isnan(neighbor_values)]

        if len(neighbor_values) > 0:
            # Use median of neighbors (robust to outliers)
            filtered[i] = np.median(neighbor_values)
        else:
            filtered[i] = values[i]

    return filtered

# Sample mu function with layer-based averaging - TWO METHODS
def sample_mu_across_fault_layered(mu_grid, coords, normals, layer_center, layer_thickness,
                                    n_samples=5, field_name='shear modulus', 
                                    method='linear'):
    """
    Sample μ on both sides of fault by averaging over a thin layer

    Parameters:
    -----------
    mu_grid : PyVista UnstructuredGrid
        Grid containing shear modulus field
    coords : ndarray (N, 3)
        Fault vertex coordinates
    normals : ndarray (N, 3)
        Fault vertex normals (pointing from FW to HW)
    layer_center : float
        Distance from fault to center of sampling layer (meters)
    layer_thickness : float
        Thickness of sampling layer (meters)
    n_samples : int
        Number of sampling points within each layer
    field_name : str
        Name of shear modulus field in mu_grid
    method : str
        'linear' - scipy linear interpolation (smooth, artificial)
        'mesh_vertices' - use actual mesh vertices (true FEniCS values)

    Returns:
    --------
    mu_hw : ndarray
        Layer-averaged μ on hanging wall side (GPa)
    mu_fw : ndarray
        Layer-averaged μ on footwall side (GPa)
    """
    from scipy.interpolate import griddata
    from scipy.spatial import cKDTree

    # Create offsets within the layer
    offset_min = layer_center - layer_thickness / 2.0
    offset_max = layer_center + layer_thickness / 2.0

    print(f"\nLayer-based μ sampling:")
    print(f"  Layer center: {layer_center/1e3:.1f} km from fault")
    print(f"  Layer thickness: {layer_thickness/1e3:.1f} km")
    print(f"  Sampling range: [{offset_min/1e3:.1f}, {offset_max/1e3:.1f}] km")
    print(f"  Number of samples per layer: {n_samples}")
    print(f"  Method: {method}")

    grid_points = mu_grid.points
    mu_values = mu_grid.point_data[field_name]

    if method == 'mesh_vertices':
        print(f"  Using ACTUAL MESH VERTICES with subdomain verification")
        print(f"  Sampling strictly along normal direction")

        # Build KDTree for mesh points
        tree = cKDTree(grid_points)

        # Get subdomain IDs for each mesh point
        # We need to query which subdomain each mesh vertex belongs to
        print(f"  Extracting subdomain IDs for mesh vertices...")

        # Create mapping from mesh coordinates to subdomain
        mesh_coords = mesh.coordinates()
        mesh_to_subdomain = {}

        for cell_idx in range(mesh.num_cells()):
            subdomain_id = subdomains[cell_idx]
            cell = dl.Cell(mesh, cell_idx)
            for vertex in dl.vertices(cell):
                vertex_idx = vertex.index()
                mesh_to_subdomain[vertex_idx] = subdomain_id

        # Now match mu_grid points to mesh vertices
        tree_mesh = cKDTree(mesh_coords)
        grid_to_subdomain = np.zeros(len(grid_points), dtype=int)

        for i, pt in enumerate(grid_points):
            dist, idx = tree_mesh.query(pt, k=1)
            if dist < 100:  # Within 100m of a mesh vertex
                if idx in mesh_to_subdomain:
                    grid_to_subdomain[i] = mesh_to_subdomain[idx]

        print(f"  Mapped {np.sum(grid_to_subdomain > 0)} grid points to subdomains")

        # Define sampling offsets
        offsets = np.linspace(offset_min, offset_max, n_samples)

        # Larger search radius but verify subdomain
        max_search_dist = 50000.0  # 50 km

        mu_hw_list = []
        mu_fw_list = []

        n_missing_hw = 0
        n_missing_fw = 0

        for i, (coord, normal) in enumerate(zip(coords, normals)):
            hw_values = []
            fw_values = []

            for offset in offsets:
                point_hw = coord + offset * normal
                point_fw = coord - offset * normal

                # Find k nearest neighbors instead of just 1
                dists_hw, indices_hw = tree.query(point_hw, k=20)
                dists_fw, indices_fw = tree.query(point_fw, k=20)

                # HW side: Find nearest point from HW subdomain (blockright)
                for dist, idx in zip(dists_hw, indices_hw):
                    if dist < max_search_dist and grid_to_subdomain[idx] == blockright:
                        hw_values.append(mu_values[idx])
                        break  # Use first valid point

                # FW side: Find nearest point from FW subdomain (blockleft)
                for dist, idx in zip(dists_fw, indices_fw):
                    if dist < max_search_dist and grid_to_subdomain[idx] == blockleft:
                        fw_values.append(mu_values[idx])
                        break  # Use first valid point

            if len(hw_values) > 0:
                mu_hw_list.append(np.mean(hw_values))
            else:
                mu_hw_list.append(np.nan)
                n_missing_hw += 1

            if len(fw_values) > 0:
                mu_fw_list.append(np.mean(fw_values))
            else:
                mu_fw_list.append(np.nan)
                n_missing_fw += 1

        mu_hw = np.array(mu_hw_list)
        mu_fw = np.array(mu_fw_list)

        # After sampling with robust_average:
        print("\nApplying spatial consistency check...")
        mu_hw = spatial_median_filter(fault_coords, mu_hw, k_neighbors=10)
        mu_fw = spatial_median_filter(fault_coords, mu_fw, k_neighbors=10)

        # 3. Fill any remaining NaN values
        print(f"\nFilling NaN values...")
        print(f"  NaN count before: HW={np.sum(np.isnan(mu_hw))}, FW={np.sum(np.isnan(mu_fw))}")

        valid_hw = ~np.isnan(mu_hw)
        if np.sum(valid_hw) > 3:
            mu_hw = griddata(fault_coords[valid_hw], mu_hw[valid_hw],
                            fault_coords, method='nearest')

        valid_fw = ~np.isnan(mu_fw)
        if np.sum(valid_fw) > 3:
            mu_fw = griddata(fault_coords[valid_fw], mu_fw[valid_fw],
                            fault_coords, method='nearest')
        print(f"  NaN count after: HW={np.sum(np.isnan(mu_hw))}, FW={np.sum(np.isnan(mu_fw))}")
    
    else:
        # METHOD: Linear interpolation (original approach)
        print(f"  Interpolation: linear (smooth interpolation between mesh points)")

        offsets = np.linspace(offset_min, offset_max, n_samples)

        mu_hw_samples = []
        mu_fw_samples = []

        for offset in offsets:
            # Points on hanging wall side (positive normal direction)
            points_hw = coords + offset * normals
            # Points on footwall side (negative normal direction)
            points_fw = coords - offset * normals

            # Interpolate μ at these points using LINEAR interpolation
            mu_hw_at_offset = griddata(grid_points, mu_values, points_hw, method='linear')
            mu_fw_at_offset = griddata(grid_points, mu_values, points_fw, method='linear')

            mu_hw_samples.append(mu_hw_at_offset)
            mu_fw_samples.append(mu_fw_at_offset)

        # Average over all samples in the layer
        mu_hw = np.nanmean(mu_hw_samples, axis=0)
        mu_fw = np.nanmean(mu_fw_samples, axis=0)

    print(f"\nLayer-averaged μ (HW): min={np.nanmin(mu_hw):.1f}, max={np.nanmax(mu_hw):.1f}, mean={np.nanmean(mu_hw):.1f} GPa")
    print(f"Layer-averaged μ (FW): min={np.nanmin(mu_fw):.1f}, max={np.nanmax(mu_fw):.1f}, mean={np.nanmean(mu_fw):.1f} GPa")

    return mu_hw, mu_fw


## 3. Load μ from XDMF Files

In [ ]:
# Function to load xdmf as pyvista grid
def load_xdmf_as_pyvista(filepath):
    """
    Load XDMF file and convert to PyVista UnstructuredGrid
    """
    import meshio
    
    # Read with meshio
    mesh_data = meshio.read(filepath)
    
    # Get points
    points = mesh_data.points
    
    # Get cells - assume tetrahedral mesh
    cells_data = []
    for cell_block in mesh_data.cells:
        cell_type = cell_block.type
        cell_data = cell_block.data
        
        # Map meshio cell type to pyvista
        if cell_type == 'tetra':
            pv_cell_type = pv.CellType.TETRA
            n_points_per_cell = 4
        elif cell_type == 'hexahedron':
            pv_cell_type = pv.CellType.HEXAHEDRON  
            n_points_per_cell = 8
        else:
            continue
            
        # Format for pyvista: [n_points, p0, p1, ..., n_points, p0, p1, ...]
        for cell in cell_data:
            cells_data.append(n_points_per_cell)
            cells_data.extend(cell)
    
    cells = np.array(cells_data)
    
    # Create cell types array
    n_cells = len(cells_data) // (n_points_per_cell + 1)
    cell_types = np.full(n_cells, pv_cell_type, dtype=np.uint8)
    
    # Create unstructured grid
    grid = pv.UnstructuredGrid(cells, cell_types, points)
    
    # Add point/cell data
    for name, data in mesh_data.point_data.items():
        grid.point_data[name] = data
    
    for name, data in mesh_data.cell_data.items():
        # Cell data comes as list of arrays (one per cell block)
        if isinstance(data, list) and len(data) > 0:
            grid.cell_data[name] = data[0]
        else:
            grid.cell_data[name] = data
    
    return grid

print("Load function defined")

In [ ]:
# Load mu grids
mu_3d_file = resultpath + f"mu_true_{meshname}{het3d_str}.xdmf"
mu_hom_file = resultpath + f"mu_true_{meshname}{homo_str}.xdmf"

print(f"Loading 3D mu from: {mu_3d_file}")
mu_3d_grid = load_xdmf_as_pyvista(mu_3d_file)

print(f"Loading homogeneous mu from: {mu_hom_file}")
mu_hom_grid = load_xdmf_as_pyvista(mu_hom_file)

# Check field names
print(f"\n3D mu fields: {list(mu_3d_grid.point_data.keys())}")
print(f"Homogeneous mu fields: {list(mu_hom_grid.point_data.keys())}")

# Print basic statistics for 3D heterogeneous model
print(f"\n3D Heterogeneous Model Statistics:")
print(f"  Mean: {mu_3d_grid['shear modulus'].mean():.2f} GPa")
print(f"  Min:  {mu_3d_grid['shear modulus'].min():.2f} GPa")
print(f"  Max:  {mu_3d_grid['shear modulus'].max():.2f} GPa")
print(f"  Std:  {mu_3d_grid['shear modulus'].std():.2f} GPa")

# Print basic statistics for homogeneous model
print(f"\nHomogeneous Model Statistics:")
print(f"  Mean: {mu_hom_grid['shear modulus'].mean():.2f} GPa")
print(f"  Min:  {mu_hom_grid['shear modulus'].min():.2f} GPa")
print(f"  Max:  {mu_hom_grid['shear modulus'].max():.2f} GPa")
print(f"  Std:  {mu_hom_grid['shear modulus'].std():.2f} GPa")

# Compute anomaly statistics
mu_3d_grid_mean = mu_3d_grid['shear modulus'].mean()
mu_hom_grid_mean = mu_hom_grid['shear modulus'].mean()

# reference mu for anomaly calculations
# mu_ref = mu_hom_grid_mean

if het3d_str == "_DeShon3D_ref_4":
    # mu_ref = 53.20    # this is mean of the orginal model, not amplified
    mu_ref = mu_3d_grid_mean
    # mu_ref = (mu_3d_grid['shear modulus'].min()+mu_3d_grid['shear modulus'].max())/2

elif het3d_str == "_mul55u25":
    # mu_ref = (mu_3d_grid['shear modulus'].min()+mu_3d_grid['shear modulus'].max())/2
    mu_ref = 40.0
else:
    mu_ref = mu_3d_grid_mean

print(f"\nUsing reference μ = {mu_ref:.2f} GPa for anomaly calculations")

mu_anomaly = (mu_3d_grid['shear modulus'] - mu_ref) / mu_ref * 100
print(f"\nAnomaly Statistics (%):")
print(f"  Mean: {mu_anomaly.mean():.2f}%")
print(f"  Min:  {mu_anomaly.min():.2f}%")
print(f"  Max:  {mu_anomaly.max():.2f}%")
print(f"  Std:  {mu_anomaly.std():.2f}%")

In [ ]:
# Extract mu at fault coordinates 
from scipy.interpolate import griddata

# Extract grid points and mu values
grid_points = mu_3d_grid.points
mu_3d_values = mu_3d_grid.point_data['shear modulus']
mu_hom_values = mu_hom_grid.point_data['shear modulus']

# Interpolate to fault coordinates
print("\n3D mu at fault:")
mu_fault_3d = griddata(grid_points, mu_3d_values, fault_coords, method='nearest')
print(f"μ: min={mu_fault_3d.min():.1f}, max={mu_fault_3d.max():.1f}, mean={mu_fault_3d.mean():.1f} GPa, n={len(mu_fault_3d)}")

print("\nHomogeneous mu at fault:")
mu_fault_hom = griddata(grid_points, mu_hom_values, fault_coords, method='nearest')
print(f"μ: min={mu_fault_hom.min():.1f}, max={mu_fault_hom.max():.1f} GPa, n={len(mu_fault_hom)}")

mu_anomaly = (mu_fault_3d - mu_ref) / mu_ref * 100
print(f"\nμ anomaly: min={mu_anomaly.min():.1f}%, max={mu_anomaly.max():.1f}%, n={len(mu_anomaly)}")


## 4. Load Slip Data

Slip files contain only (s_strike, s_dip) columns. Coordinates come from fault_geometry file.

**For locking problems:**
- Raw slip files are stored in meters (physical units from forward/inverse modeling)
- Must normalize by `amp` (trench-normal loading rate) to get **coupling ratio**
- Coupling ratio: 0 = free slip, 1 = complete locking (plate velocity)
- This normalization is applied to ALL slip files (ground truth AND inferred)

In [ ]:
# Load fault geometry (coordinates)

print(f"Fault geometry loaded: {len(fault_geom)} nodes")
print(f"  x: [{fault_geom['x'].min()/1e3:.3f}, {fault_geom['x'].max()/1e3:.3f}] km")
print(f"  y: [{fault_geom['y'].min()/1e3:.3f}, {fault_geom['y'].max()/1e3:.3f}] km")
print(f"  z: [{fault_geom['z'].min()/1e3:.3f}, {fault_geom['z'].max()/1e3:.3f}] km")


In [ ]:
# Function to read slip files
def read_slip_file(filename, fault_geom, is_ground_truth=False, problem_type='locking', amp=None):
    """
    Read slip file containing (s_strike, s_dip) and combine with fault geometry
    
    Parameters:
    -----------
    filename : str
        Path to slip file
    fault_geom : DataFrame
        Fault geometry with (x, y, z) coordinates
    is_ground_truth : bool
        If True, mark as ground truth (affects print messages)
    problem_type : str
        'coseismic' or 'locking' - determines normalization
    amp : float
        Amplitude for normalization (if problem_type='locking')
        For locking: slip files are in meters, normalize by amp to get coupling ratio
        
    Returns:
    --------
    DataFrame with columns: x, y, z, s_strike, s_dip, slip_mag
    """
    # Read slip values (only 2 columns: s_strike, s_dip)
    slip_data = pd.read_csv(filename, sep=r'\s+', header=None,
                           names=['s_strike', 's_dip'])
    
    # Combine with geometry
    result = fault_geom.copy()
    result['s_strike'] = slip_data['s_strike'].values
    result['s_dip'] = slip_data['s_dip'].values
    
    # Compute slip magnitude
    result['slip_mag'] = np.sqrt(result['s_strike']**2 + result['s_dip']**2)
    
    # Apply normalization for locking problems (ALL files, not just ground truth)
    if problem_type == 'locking':
        if amp is None:
            raise ValueError("amp parameter required for locking problem normalization")
        cols_to_convert = ['s_strike', 's_dip', 'slip_mag']
        result[cols_to_convert] = result[cols_to_convert] / amp
        file_type = "ground truth" if is_ground_truth else "inferred"
        print(f"  Applied locking normalization ({file_type}, divided by amp={amp:.4f})")
    
    print(f"  Loaded: {len(result)} nodes")
    print(f"  Slip range: [{result['slip_mag'].min():.4f}, {result['slip_mag'].max():.4f}]")
    print(f"  s_dip range: [{result['s_dip'].min():.4f}, {result['s_dip'].max():.4f}]")
    
    return result

print(f"Slip loading function defined (default: problem_type='locking')")

In [ ]:
# # Load true slip
# print("Loading true slip (ground truth)...")
# mtrue_file = resultpath + f"mtrue_s_fault_{meshname}{slip_str_gt}.txt"
# mtrue_s = read_slip_file(mtrue_file, fault_geom, is_ground_truth=True, 
#                          problem_type=problem_type, amp=amp)
# print(mtrue_s.head())

In [ ]:
# # Load inferred slip - 3D inversion
# print("\nLoading 3D inversion slip...")
# m_3d_file = resultpath + f"m_s_fault_{meshname}{slip_str_gt}{het3d_str}{inv_str}{het3d_str}.txt"
# m_s_3d = read_slip_file(m_3d_file, fault_geom, is_ground_truth=False, 
#                         problem_type=problem_type, amp=amp)

In [ ]:
# # Load inferred slip - homogeneous inversion
# print("\nLoading homogeneous inversion slip...")
# m_hom_file = resultpath + f"m_s_fault_{meshname}{slip_str_gt}{het3d_str}{inv_str}{homo_str}.txt"
# m_s_hom = read_slip_file(m_hom_file, fault_geom, is_ground_truth=False,
#                          problem_type=problem_type, amp=amp)

# # Compute slip difference (3D - hom): positive means hom underestimates
# # For locking problems, use s_dip (downdip component) which represents coupling
# # Store in m_s_3d since it represents deviation from the "correct" 3D result
# if problem_type == 'locking':
#     m_s_3d['slip_diff'] = m_s_3d['s_dip'] - m_s_hom['s_dip']
#     slip_component = 's_dip (downdip coupling)'
# else:
#     m_s_3d['slip_diff'] = m_s_3d['slip_mag'] - m_s_hom['slip_mag']
#     slip_component = 'slip magnitude'

# print(f"\nSlip difference (3D - hom), using {slip_component}:")
# print(f"  Mean: {m_s_3d['slip_diff'].mean():.4f}")
# print(f"  RMS: {np.sqrt((m_s_3d['slip_diff']**2).mean()):.4f}")
# print(f"  Range: [{m_s_3d['slip_diff'].min():.4f}, {m_s_3d['slip_diff'].max():.4f}]")
# print(f"  Interpretation: positive = hom underestimates, negative = hom overestimates")

In [ ]:
# Add lon, lat, depth for PyGMT plotting
# Convert from local coordinates (x, y, z in meters) to geographic (lon, lat, depth in km)

# Import utility for coordinate transformation (consistent with mesh generation)
import sys
sys.path.append('/home/staff/chao/SSEinv/Nicoya/codes/')
import utils_plot as utp

# Mesh-specific transformation parameters (nicoyaCK4)
# These match the parameters used when creating the mesh
lon0, lat0 = -84, 7      # Reference point (center of projection), from Christos's email
rot = 45                  # Rotation angle in degrees, positive is CCW
x0, y0 = 130e3, 350e3    # Offset for x and y coordinates in meters

# Note: For reverse conversion (lon,lat → x,y), use:
# x_rot, y_rot = utp.LL2ckmd(lon, lat, lon0, lat0, rot)
# Then subtract offsets: x = x_rot - x0, y = y_rot - y0

# Also add geographic coordinates to fault facet centers
fault_lon, fault_lat = utp.ckm2LLd(fault_coords[:, 0] + x0, 
                                    fault_coords[:, 1] + y0, 
                                    lon0, lat0, -rot)
fault_depth = -fault_coords[:, 2] / 1000.0  # Convert z to depth in km

print(f"\nGeographic coordinates added to fault facet centers:")
print(f"  Lon: [{fault_lon.min():.3f}, {fault_lon.max():.3f}]°")
print(f"  Lat: [{fault_lat.min():.3f}, {fault_lat.max():.3f}]°")
print(f"  Depth: [{fault_depth.min():.1f}, {fault_depth.max():.1f}] km")

In [ ]:
# def add_geographic_coords(df):
#     """
#     Add lon, lat, depth columns to dataframe with x, y, z
#     Uses the same coordinate transformation as mesh generation
#     """
#     # Convert using custom transformation (consistent with mesh)
#     # Note: rotation is negated (-rot) to match mesh convention
#     df['lon'], df['lat'] = utp.ckm2LLd(df['x'].values + x0, 
#                                         df['y'].values + y0, 
#                                         lon0, lat0, -rot)
#     df['depth'] = -df['z'].values / 1000.0  # Convert z (meters, negative down) to depth (km, positive down)
#     return df

# # Add geographic coordinates to all slip dataframes
# mtrue_s = add_geographic_coords(mtrue_s)
# m_s_3d = add_geographic_coords(m_s_3d)
# m_s_hom = add_geographic_coords(m_s_hom)

# print(f"\nGeographic coordinates added to slip dataframes (using mesh-consistent transformation):")
# print(f"  Reference: lon0={lon0}°, lat0={lat0}°, rot={rot}°")
# print(f"  Offset: x0={x0/1e3:.0f} km, y0={y0/1e3:.0f} km")
# print(f"  Lon: [{m_s_3d['lon'].min():.3f}, {m_s_3d['lon'].max():.3f}]°")
# print(f"  Lat: [{m_s_3d['lat'].min():.3f}, {m_s_3d['lat'].max():.3f}]°")
# print(f"  Depth: [{m_s_3d['depth'].min():.1f}, {m_s_3d['depth'].max():.1f}] km")


## 5. Load Displacement Data at Stations

In [ ]:
# Coordinate rotation function
def rot_xy(x, y, rot):
    cos_rot = np.cos(np.radians(rot))
    sin_rot = np.sin(np.radians(rot))
    x_rot = x * cos_rot + y * sin_rot
    y_rot = -x * sin_rot + y * cos_rot
    return x_rot, y_rot

print("Coordinate rotation function defined")

In [ ]:
# # Load observation file
# d_obs_file = resultpath + f"d_obs_{meshname}{slip_str_gt}{het3d_str}.txt"
# df_obs = pd.read_csv(d_obs_file, sep=r'\s+', header=None,
#                      names=['x', 'y', 'z', 'ux', 'uy', 'uz'])

# # Reverse rotation to geographic coordinates
# df_obs['ux'], df_obs['uy'] = rot_xy(df_obs['ux'], df_obs['uy'], -rot)

# # Load actual GNSS station locations
# obs_disp_name = "CKfig6_data_final.csv"
# gnss_stations = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1,
#                             names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car',
#                                    'vx_std_Car', 'vy_std_Car', 'vz_std_Car'])

# df_obs['lon'] = gnss_stations['lon'].values
# df_obs['lat'] = gnss_stations['lat'].values
# df_obs['depth'] = -df_obs['z'].values / 1000.0
# df_obs['u_h'] = np.sqrt(df_obs['ux']**2 + df_obs['uy']**2)

# print(f"Loaded displacement at {len(df_obs)} stations")
# print(f"Horizontal displacement range: {df_obs['u_h'].min():.4f} - {df_obs['u_h'].max():.4f} m")

## 6. Visualize μ on Fault Surface

**Spatial smoothing for visualization:**
- Applies Gaussian smoothing to reduce noise in plots while preserving data for analysis
- Smoothing is done via triangulation-based interpolation onto a regular grid
- Original data remains unchanged for correlation analysis

In [ ]:
import cmcrameri.cm as cmc
from scipy.ndimage import gaussian_filter
from scipy.interpolate import griddata

# Helper function for spatial smoothing
def smooth_fault_data(coords, values, sigma=2.0, grid_spacing=1000.0):
    """
    Apply spatial smoothing to fault data for visualization
    
    Parameters:
    -----------
    coords : ndarray (N, 3)
        Fault vertex coordinates (x, y, z)
    values : ndarray (N,)
        Values to smooth (e.g., mu, delta_mu)
    sigma : float
        Gaussian kernel standard deviation (in grid cells)
        Larger = more smoothing. Typical: 1-3
    grid_spacing : float
        Regular grid spacing in meters. Default 1000m = 1km
    
    Returns:
    --------
    coords_smooth : ndarray
        Coordinates of smoothed data (for plotting)
    values_smooth : ndarray
        Smoothed values
    """
    # Create regular grid
    x_min, x_max = coords[:, 0].min(), coords[:, 0].max()
    y_min, y_max = coords[:, 1].min(), coords[:, 1].max()
    
    xi = np.arange(x_min, x_max, grid_spacing)
    yi = np.arange(y_min, y_max, grid_spacing)
    xi_grid, yi_grid = np.meshgrid(xi, yi)
    
    # Interpolate to regular grid
    values_grid = griddata(coords[:, :2], values, (xi_grid, yi_grid), method='linear')
    
    # Apply Gaussian smoothing
    values_smooth_grid = gaussian_filter(values_grid, sigma=sigma, mode='nearest')
    
    # Flatten for plotting
    mask = ~np.isnan(values_smooth_grid)
    x_smooth = xi_grid[mask]
    y_smooth = yi_grid[mask]
    values_smooth = values_smooth_grid[mask]
    
    coords_smooth = np.column_stack([x_smooth, y_smooth])
    
    return coords_smooth, values_smooth

# Plot mu on fault surface using matplotlib
def plot_fault_mu_matplotlib(fault_coords, mu_values, title="μ on Fault Surface", cmap='viridis', 
                             symmetric=False, clabel='μ (GPa)', smooth=False, sigma=2.0):
    """
    Plot mu on fault surface
    
    Parameters:
    -----------
    smooth : bool
        If True, apply spatial smoothing for visualization
    sigma : float
        Smoothing strength (larger = more smoothing)
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    if smooth:
        # Apply smoothing for visualization
        coords_smooth, mu_smooth = smooth_fault_data(fault_coords, mu_values, sigma=sigma)
        x_km = coords_smooth[:, 0] / 1e3
        y_km = coords_smooth[:, 1] / 1e3
        plot_values = mu_smooth
        title = title + f" (smoothed, σ={sigma})"
    else:
        x_km = fault_coords[:, 0] / 1e3
        y_km = fault_coords[:, 1] / 1e3
        plot_values = mu_values

    if symmetric:
        vmax = np.nanmax(np.abs(plot_values))
        vmin = -vmax
    else:
        vmin, vmax = None, None

    sc = ax.scatter(x_km, y_km, c=plot_values, s=50,
                    cmap=cmap, vmin=vmin, vmax=vmax, alpha=0.8)

    ax.set_xlabel('X (km)', fontsize=12)
    ax.set_ylabel('Y (km)', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(clabel, fontsize=12)

    plt.tight_layout()
    return fig, ax

# # Usage: Compare with and without smoothing
# print("Without smoothing:")
# fig1 = plot_fault_mu_matplotlib(fault_coords, mu_fault_3d, "μ (3D)", cmap='gist_rainbow_r', smooth=False)
# # parint("\nWith smoothing (σ=2):")
# # fig2 = plot_fault_mu_matplotlib(fault_coords, mu_fault_3d, "μ (3D)", cmap='gist_rainbow_r', smooth=True, sigma=2.0)
# # print("\nWith smoothing (σ=3):")
# fig3 = plot_fault_mu_matplotlib(fault_coords, mu_anomaly, "μ Anomaly (relative to homogeneous)", 
#                                 cmap='cmc.roma', symmetric=True, clabel='Anomly (%)', smooth=False)

## 7. Slip Comparison (PyGMT)

In [ ]:
# Read the mesh file for generating the slab interface
import meshio
mesh_file = "Kyriakopoulos2016JGR/Nicoya_interface.e"
plate_mesh = meshio.read("/home/staff/chao/SSEinv/Nicoya/" + mesh_file)
points = plate_mesh.points  # shape (n_points, 3)
df_plate = pd.DataFrame(points, columns=["x", "y", "z"])
# define the centroid of relative coordinates that is consistent with mesh
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
df_plate['lon'], df_plate['lat'] = utp.ckm2LLd(df_plate['x'], df_plate['y'], lon0, lat0, -rot)
df_plate['z'] = df_plate['z'] /1e3  
# print(df_plate.head())

# Read the trench file
# trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_geo.txt"
# trench = pd.read_csv(trench_file, sep=r'\s+', names=['lon', 'lat'])
trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_xyz.txt"
trench = pd.read_csv(trench_file, sep=r'\s+', names=['x', 'y'])
trench['lon'], trench['lat'] = utp.ckm2LLd(trench['x'], trench['y'], lon0, lat0, -rot)
# print(trench.head())

m2mm = 1e3  # meters to millimeters conversion factor

region=[-87, -84, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
# region=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
# region=[-88, -83, 6, 14]    # suitable region for chopping the plate interface grid file 

# noise_std_h = 0.5 * (gnss_stations['vx_std_Car'].mean() + gnss_stations['vy_std_Car'].mean()) 
# noise_std_v = gnss_stations['vz_std_Car'].mean()
# error_e, error_n, error_v  = noise_std_h, noise_std_h, noise_std_v

In [ ]:
# def plot_slip_comparison_pygmt(m_3d, m_hom, col2plt='s_dip', title_suffix='Coupling', cmap='viridis'):
#     """Plot 3-panel slip comparison: 3D inv, Hom inv, Difference"""

#     # Define region
#     # region = [m_3d['lon'].min()-0.3, m_3d['lon'].max()+0.3,
#     #         m_3d['lat'].min()-0.3, m_3d['lat'].max()+0.3]

#     fig = pygmt.Figure()

#     with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False,
#                     frame=["a1f0.2", "WSne"], margins=["0.1c", "0.2c"], sharey=True):

#         pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
#                     MAP_TITLE_OFFSET="-0.2c", FONT_ANNOT_PRIMARY="6p")

#         # Common settings
#         maxval = max(m_3d[col2plt].max(), m_hom[col2plt].max())
#         maxval = 1.0
#         print(maxval)
        
#         style = "c0.09c"

#         # Panel 1: 3D inversion
#         with fig.set_panel(panel=[0, 0]):
#             fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+t3D inv."])
#             pygmt.makecpt(cmap=cmap, series=[0, maxval, maxval/20], reverse=True)
#             fig.plot(x=m_3d['lon'], y=m_3d['lat'], style=style, fill=m_3d[col2plt], cmap=True)
#             fig.coast(shorelines="0.25p,black", area_thresh=20)
#             with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
#                 fig.colorbar(frame=["af", f"x+l{title_suffix}"])
#             grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'],
#                                  region=region, spacing=(0.04, 0.04))
#             fig.grdcontour(grid=grid, levels=5, limit="-100/-5",
#                           annotation="20+f8p", pen="0.25p,darkgray")
#             fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray",
#                     style="f0.4i/0.075i+l+t", fill="dimgray")
            
#         # Panel 2: Homogeneous inversion
#         with fig.set_panel(panel=[0, 1]):
#             fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHom. inv."])
#             pygmt.makecpt(cmap=cmap, series=[0, maxval, maxval/20], reverse=True)
#             fig.plot(x=m_hom['lon'], y=m_hom['lat'], style=style, fill=m_hom[col2plt], cmap=True)
#             fig.coast(shorelines="0.25p,black", area_thresh=20)
#             with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
#                 fig.colorbar(frame=["af", f"x+l{title_suffix}"])
#             grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'],
#                                  region=region, spacing=(0.04, 0.04))
#             fig.grdcontour(grid=grid, levels=5, limit="-100/-5",
#                           annotation="20+f8p", pen="0.25p,darkgray")
#             fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray",
#                     style="f0.4i/0.075i+l+t", fill="dimgray")
            
#         # Panel 3: Difference
#         with fig.set_panel(panel=[0, 2]):
#             fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+t3D - Hom"])
#             diff = m_3d[col2plt] - m_hom[col2plt]
#             maxdiff = diff.abs().max()
#             maxdiff = 0.6
#             pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], reverse=False)
#             fig.plot(x=m_3d['lon'], y=m_3d['lat'], style=style, fill=diff, cmap=True)
#             fig.coast(shorelines="0.25p,black", area_thresh=20)
#             with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
#                 fig.colorbar(frame=["af", "x+lDifference"])
#             grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'],
#                                  region=region, spacing=(0.04, 0.04))
#             fig.grdcontour(grid=grid, levels=5, limit="-100/-5",
#                           annotation="20+f8p", pen="0.25p,darkgray")
#             fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray",
#                     style="f0.4i/0.075i+l+t", fill="dimgray")
            
#     return fig

# # Usage
# if problem_type == 'locking':
#     fig_slip_compare = plot_slip_comparison_pygmt(m_s_3d, m_s_hom, col2plt='s_dip',
#                                                     title_suffix='Coupling', cmap='viridis')
# else:
#     fig_slip_compare = plot_slip_comparison_pygmt(m_s_3d, m_s_hom, col2plt='slip_mag',
#                                                     title_suffix='Slip (m)', cmap='viridis')

# fig_slip_compare.show()

# if flag_savefig:
#     fig_slip_compare.savefig(resultpath + 'slip_comparison_3panel.png', dpi=300)

## 8. Displacement at Stations (PyGMT)

In [ ]:
# # Add helper function
# def build_disp_vector(u_true, scaleunit, error_e=None, error_n=None, error_v=None):
#     # Horizontal components
#     df_obs_h_data = {
#         "x": u_true['lon'],
#         "y": u_true['lat'],
#         "east_velocity": u_true['ux']*scaleunit,
#         "north_velocity": u_true['uy']*scaleunit,
#     }

#     # Add error columns only if errors are provided
#     if error_e is not None and error_n is not None:
#         df_obs_h_data["east_sigma"] = error_e*scaleunit
#         df_obs_h_data["north_sigma"] = error_n*scaleunit
#         df_obs_h_data["correlation_EN"] = 0.0

#     df_obs_h = pd.DataFrame(data=df_obs_h_data)

#     # Vertical components
#     df_obs_v_data = {
#         "x": u_true['lon'],
#         "y": u_true['lat'],
#         "east_velocity": 0.0,
#         "north_velocity": u_true['uz']*scaleunit,
#     }

#     # Add error columns only if errors are provided
#     if error_v is not None:
#         df_obs_v_data["east_sigma"] = error_v*scaleunit
#         df_obs_v_data["north_sigma"] = error_v*scaleunit
#         df_obs_v_data["correlation_EN"] = 0.0

#     df_obs_v = pd.DataFrame(data=df_obs_v_data)

#     return df_obs_h, df_obs_v

In [ ]:
# def plot_slip_and_displacement(mtrue_s, df_obs, trench, df_plate, region, error_e=None, error_n=None, error_v=None, scale_vec=500):
#     m2mm = 1000
#     col2plt = 's_dip'
    
#     df_obs_h, df_obs_v = build_disp_vector(df_obs, m2mm, error_e, error_n, error_v)
    
#     fig = pygmt.Figure()
    
#     with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False,
#                      frame=["a1f0.2", "WSne"], margins=["0.1c", "0.2c"], sharey=True):
        
#         pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black",
#                      MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
#                      FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")
        
#         # Panel 1: True slip
#         with fig.set_panel(panel=[0, 0]):
#             fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
#             maxslip = 1 * amp * m2mm
#             pygmt.makecpt(cmap="viridis", series=[0, maxslip, maxslip/10], background=True, reverse=True)
#             fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style="c0.08c",
#                      fill=mtrue_s[col2plt]*amp*m2mm, cmap=True)
            
#             scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
#             mapscale_str = f"g{scale_lon}/{scale_lat}c{scale_lat}+w50k"
#             fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            
#             grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'],
#                                  region=region, spacing=(0.04, 0.04))
#             fig.grdcontour(grid=grid, levels=5, limit="-100/-5",
#                           annotation="20+f8p", pen="0.5p,darkgray")
#             fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray",
#                     style="f0.4i/0.075i+l+t", fill="dimgray")
#             fig.plot(x=df_obs['lon'], y=df_obs['lat'], style="s0.15c",
#                     fill="cyan", pen="0.25p,black")
            
#             with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
#                 fig.colorbar(frame=["af", "x+lSlip mag.", "y+l(mm)"])
        
#         # Panel 2: Horizontal displacement
#         with fig.set_panel(panel=[0, 1]):
#             fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic horiz. disp."])
#             fig.coast(shorelines="0.25p,black", area_thresh=20, land="lightgray",
#                      water=None, resolution="h", map_scale=mapscale_str)
            
#             grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'],
#                                 region=region, spacing=(0.04, 0.04))
#             fig.grdcontour(grid=grid, levels=5, limit="-100/-5",
#                           annotation="20+f8p", pen="0.5p,darkgray")
#             fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray",
#                     style="f0.4i/0.075i+l+t", fill="dimgray")
            
#             fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black",
#                     spec=f"e{0.5/scale_vec}/0.39",
#                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            
#             # Legend
#             fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2",
#                     fill="white", pen="0.5p,black", transparency=30)
#             fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c",
#                     fill="cyan", pen="0.25p,black")
            
#             lgdobs = pd.DataFrame({
#                 "x": [region[0]+0.15], "y": [region[-2]+0.5],
#                 "east_velocity": [1], "north_velocity": [0],
#                 "east_sigma": [1/5], "north_sigma": [1/5], "correlation_EN": [0.0]
#             })
#             fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black",
#                     spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            
#             fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8,
#                     font="8p,Helvetica,black", justify="ML")
#             fig.text(text="Obs H", x=region[0]+0.6, y=region[-2]+0.5,
#                     font="8p,Helvetica,black", justify="ML")
#             fig.text(text=f"{int(scale_vec)}±{int(scale_vec/5)} mm",
#                     x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")
        
#         # Panel 3: Vertical displacement
#         with fig.set_panel(panel=[0, 2]):
#             fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic vert. disp."])
#             fig.coast(shorelines="0.25p,black", area_thresh=20, land="lightgray",
#                      water=None, resolution="h", map_scale=mapscale_str)
            
#             grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'],
#                                 region=region, spacing=(0.04, 0.04))
#             fig.grdcontour(grid=grid, levels=5, limit="-100/-5",
#                           annotation="20+f8p", pen="0.5p,darkgray")
#             fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray",
#                     style="f0.4i/0.075i+l+t", fill="dimgray")
            
#             fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black",
#                     spec=f"e{0.5/scale_vec}/0.39",
#                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            
#             # Legend
#             fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2",
#                     fill="white", pen="0.5p,black", transparency=30)
#             fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c",
#                     fill="cyan", pen="0.25p,black")
            
#             lgdobs = pd.DataFrame({
#                 "x": [region[0]+0.4], "y": [region[-2]+0.3],
#                 "east_velocity": [0], "north_velocity": [1],
#                 "east_sigma": [1/5], "north_sigma": [1/5], "correlation_EN": [0.0]
#             })
#             fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black",
#                     spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            
#             fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8,
#                     font="8p,Helvetica,black", justify="ML")
#             fig.text(text="Obs V", x=region[0]+0.6, y=region[-2]+0.5,
#                     font="8p,Helvetica,black", justify="ML")
#             fig.text(text=f"{int(scale_vec)}±{int(scale_vec/5)} mm",
#                     x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")
    
#     return fig

# u_h_max = df_obs['u_h'].max() * 1000
# n = np.ceil(u_h_max / 10)
# scale_vec = np.round(5*n)
# scale_vec = 20
# print(u_h_max, scale_vec)

# fig = plot_slip_and_displacement(mtrue_s, df_obs, trench, df_plate, region, 
#                                   error_e, error_n, error_v, scale_vec)
# fig.show()
# if flag_savefig:
#      fig.savefig(resultpath + 'slip_displacement_3panel.png', dpi=300)

## 9. Combined Multi-Panel Figure

In [ ]:
# # Combined multi-panel figure (PyGMT) with panels (a)-(d) implemented
# lon_stack = np.concatenate([
#     np.asarray(fault_lon, dtype=float),
#     m_s_3d['lon'].values.astype(float),
#     df_obs['lon'].values.astype(float)
# ])
# lat_stack = np.concatenate([
#     np.asarray(fault_lat, dtype=float),
#     m_s_3d['lat'].values.astype(float),
#     df_obs['lat'].values.astype(float)
# ])
# pad_deg = 0.25
# region_fault = [
#     float(lon_stack.min() - pad_deg),
#     float(lon_stack.max() + pad_deg),
#     float(lat_stack.min() - pad_deg),
#     float(lat_stack.max() + pad_deg)
# ]

region_fault = region

plate_grd = pygmt.xyz2grd(
    x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'],
    region=region_fault, spacing=(0.05, 0.05)
)

# if problem_type == 'locking':
#     slip_diff_label_short = 'Δ Coupling'
#     slip_diff_label_long = 'Coupling Difference (3D - Hom)'
#     slip_diff_label = 'Δ Coupling, mean removed'
# else:
#     slip_diff_label_short = 'Δ Slip'
#     slip_diff_label_long = 'Slip Difference (3D - Hom)'

# slip_diff_values = m_s_3d['slip_diff'].values

# scaleunit_mm = 1000.0
# # Build horizontal displacement vectors (vertical array used directly below)
# df_obs_h_panel, _ = build_disp_vector(df_obs, scaleunit_mm, error_e, error_n, error_v)

# # Use scale_vec_panel = 20 as user specified
# scale_vec_panel = 20

# vertical_mm = df_obs['uz'].values * scaleunit_mm
# vert_vmax = max(float(np.nanmax(np.abs(vertical_mm))), 1.0)
# lon_center = 0.5 * (region_fault[0] + region_fault[1])
# lat_center = 0.5 * (region_fault[2] + region_fault[3])

# fig = pygmt.Figure()

# # Shared helpers for subplot panels
# def _panel_basemap(title):
#     fig.basemap(region=region_fault, projection="M?", frame=["a1f0.2", f"+t{title}"])

# def _panel_overlays(fill_land=False):
#     coast_kwargs = dict(
#         shorelines="0.25p,black",
#         area_thresh=20,
#         land="lightgray" if fill_land else None,
#         water=None,
#         resolution="h"
#     )
#     fig.coast(**coast_kwargs)
#     fig.grdcontour(grid=plate_grd, levels=5, limit="-100/-5",
#                    annotation="20+f6p", pen="0.4p,darkgray")
#     fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray")

# def _panel_placeholder(title, message):
#     _panel_basemap(title)
#     _panel_overlays(fill_land=False)
#     fig.text(x=lon_center, y=lat_center, text=message,
#              font="9p,Helvetica,gray35", justify="CM")

# # Use same style as slip plots
# style_fault = "c0.11c"

# with fig.subplot(nrows=2, ncols=3, figsize=("21c", "15c"), autolabel=False,
#                  sharex=True, sharey=True, margins="0.45c/0.6c"):
#     # Fix title offset to prevent overlap with tick labels
#     pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
#                  MAP_TITLE_OFFSET="0.3c", FONT_ANNOT_PRIMARY="7p")

#     # Panel (a): μ on the fault from the 3D model
#     with fig.set_panel(panel=[0, 0]):
#         _panel_basemap("(a) μ on Fault (3D)")
#         mu_min = float(np.nanmin(mu_fault_3d))
#         mu_max = float(np.nanmax(mu_fault_3d))
#         mu_range = max(mu_max - mu_min, 1e-3)
#         mu_step = mu_range / 20.0
#         print(f"μ range: [{mu_min:.3f}, {mu_max:.3f}] GPa, step: {mu_step:.4f} GPa")

#         # Discontinuous colorbar with 20 bins, just like plot_slip_comparison_pygmt
#         pygmt.makecpt(cmap="rainbow", series=[mu_min, mu_max, mu_step], background=True, reverse=False)
#         fig.plot(x=fault_lon, y=fault_lat, style=style_fault,
#                  fill=mu_fault_3d, cmap=True)
#         _panel_overlays(fill_land=False)
#         # Increased colorbar tick label size from 8p to 9p
#         with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
#             fig.colorbar(frame=["af", "x+lμ (GPa)"])

#     # Panel (b): μ anomaly (% difference relative to homogeneous)
#     with fig.set_panel(panel=[0, 1]):
#         _panel_basemap("(b) μ Anomaly (%)")

#         anom_min = float(np.nanmin(mu_anomaly))
#         anom_max = float(np.nanmax(mu_anomaly))
#         print(f"Anomaly range: [{anom_min:.3f}, {anom_max:.3f}] %")

#         # mu_anomaly_rmean = (mu_fault_3d - np.mean(mu_fault_3d)) / np.mean(mu_fault_3d) * 100
#         mu_anomaly_rmean = mu_anomaly - np.mean([anom_min, anom_max])

#         mu_anomaly_plt = mu_anomaly
#         # mu_anomaly_plt = mu_anomaly_rmean

#         anom_vmax = max(float(np.nanmax(np.abs(mu_anomaly_plt))), 1e-2)
#         anom_step = anom_vmax / 10.0
#         print(f"Anomaly range: ±{anom_vmax:.3f} %, step: {anom_step:.4f} %")

#         # Discontinuous colorbar with 20 bins
#         pygmt.makecpt(cmap="roma", series=[-anom_vmax, anom_vmax, anom_step], background=True, reverse=False)
#         fig.plot(x=fault_lon, y=fault_lat, style=style_fault,
#                  fill=mu_anomaly_plt, cmap=True)
#         _panel_overlays(fill_land=False)
#         # Increased colorbar tick label size from 8p to 9p
#         with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
#             fig.colorbar(frame=["af", "x+lAnomaly (%)"])

#     # Panel (c): slip/coupling bias (3D - homogeneous)
#     with fig.set_panel(panel=[0, 2]):
#         _panel_basemap(f"(c) {slip_diff_label_long}")

#         slip_diff_min = float(np.nanmin(slip_diff_values))
#         slip_diff_max = float(np.nanmax(slip_diff_values))
#         print(f"Slip diff range: [{slip_diff_min:.3f}, {slip_diff_max:.3f}]")

#         slip_diff_values_dmean = slip_diff_values - np.mean([slip_diff_min, slip_diff_max])

#         # slip_diff_plt = slip_diff_values
#         slip_diff_plt = slip_diff_values_dmean
    
#         slip_vmax = max(float(np.nanmax(np.abs(slip_diff_plt))), 1e-4)
#         # slip_vmax = 0.52
#         if slip_diff_plt is slip_diff_values:
#             slip_vmax = 0.6
#         elif slip_diff_plt is slip_diff_values_dmean:
#             slip_vmax = 0.5

#         slip_step = slip_vmax / 10.0
#         print(f"Anomaly range: ±{slip_vmax:.3f} %, step: {slip_step:.4f}")

#         # Discontinuous colorbar with 20 bins
#         pygmt.makecpt(cmap="roma", series=[-slip_vmax, slip_vmax, slip_step], background=True, reverse=False)
#         fig.plot(x=m_s_3d['lon'], y=m_s_3d['lat'], style=style_fault,
#                  fill=slip_diff_plt, cmap=True)
#         _panel_overlays(fill_land=False)
#         # Increased colorbar tick label size from 8p to 9p
#         with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
#             if slip_diff_plt is slip_diff_values:
#                 fig.colorbar(frame=["af", f"x+l{slip_diff_label_short}"])
#             elif slip_diff_plt is slip_diff_values_dmean:
#                 fig.colorbar(frame=["af", f"x+l{slip_diff_label}"])

#     # Panel (d): synthetic surface displacement vectors + vertical signal
#     with fig.set_panel(panel=[1, 0]):
#         _panel_basemap("(d) Surface Displacement")
#         vert_step = vert_vmax / 10.0
#         # Discontinuous colorbar with 20 bins
#         pygmt.makecpt(cmap="vik", series=[-vert_vmax, vert_vmax, vert_step], background=True, reverse=True)
#         _panel_overlays(fill_land=True)
#         fig.plot(x=df_obs['lon'], y=df_obs['lat'], style="s0.18c",
#                  fill=vertical_mm, cmap=True, pen="0.25p,black")
#         fig.velo(data=df_obs_h_panel, pen="0.4p,black", line="0.25p,black",
#                  spec=f"e{0.5/scale_vec_panel}/0.39",
#                  vector="0.1c+a45+p0.75p,black+ea+gblack+h0")
#         # Increased colorbar tick label size from 8p to 9p
#         with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
#             fig.colorbar(frame=["af", "x+lVertical (mm)"])
        
#         # Legend - place at bottom left corner, similar to Cell 29
#         legend_x = region_fault[0] + 0.15
#         legend_y = region_fault[2] + 0.35  # Bottom of panel + small offset
        
#         # White box background for legend
#         fig.plot(x=legend_x + 0.45, y=legend_y + 0.1, style="r2.3/1.5",
#                  fill="white", pen="0.4p,black", transparency=20)
        
#         # Station symbol
#         fig.plot(x=legend_x + 0.15, y=legend_y + 0.35, style="s0.15c",
#                  fill="cyan", pen="0.25p,black")
        
#         # Example displacement vector
#         legend_vec = pd.DataFrame({
#             "x": [legend_x + 0.05],
#             "y": [legend_y + 0.15],
#             "east_velocity": [scale_vec_panel],
#             "north_velocity": [0.0],
#             "east_sigma": [scale_vec_panel / 5.0],
#             "north_sigma": [scale_vec_panel / 5.0],
#             "correlation_EN": [0.0]
#         })
#         fig.velo(data=legend_vec, pen="0.4p,black", line="0.25p,black",
#                  spec=f"e{0.5/scale_vec_panel}/0.39",
#                  vector="0.1c+a45+p0.75p,black+ea+gblack+h0")
        
#         # Legend text
#         fig.text(text="Stations", x=legend_x + 0.35, y=legend_y + 0.35,
#                  font="7p,Helvetica,black", justify="ML")
#         fig.text(text=f"Obs H", x=legend_x + 0.35, y=legend_y + 0.15,
#                  font="7p,Helvetica,black", justify="ML")
#         fig.text(text=f"{int(scale_vec_panel)}±{int(scale_vec_panel/5)} mm",
#                  x=legend_x + 0.05, y=legend_y - 0.05,
#                  font="7p,Helvetica,black", justify="ML")

#     # Panel (e): placeholder for Δμ map once contrasts are computed later in the notebook
#     with fig.set_panel(panel=[1, 1]):
#         _panel_placeholder("(e) μ Contrast Across Fault (TBD)",
#                            "Δμ panel will be populated with PyGMT once computed.")

#     # Panel (f): placeholder for correlation / stats panel (still Matplotlib-based)
#     with fig.set_panel(panel=[1, 2]):
#         _panel_placeholder("(f) Correlation / Binned Stats",
#                            "Structure-slip correlation plots stay in Matplotlib.")

# fig.show()
# if flag_savefig:
#     fig.savefig(resultpath + 'slip_mu_relation_combined.png', dpi=300)
# print(f"Figure saved to: {resultpath}slip_mu_relation_combined.png")

## 10. Δμ Across Fault Calculation

Now we implement the calculation of μ contrast across the fault:
- Hanging wall (HW) = overriding plate = blockright
- Footwall (FW) = subducting plate = blockleft
- Δμ = μ_HW - μ_FW

**Layer-based sampling approach:**
Instead of sampling μ at a single point on each side of the fault, we sample multiple points within a thin layer parallel to the fault interface and compute the depth-averaged modulus. This provides:
1. More robust estimates less sensitive to local heterogeneity
2. Better representation of the effective μ structure affecting Green's functions
3. Volume averaging rather than point sampling

**Implementation:**
- Define a thin layer on each side (e.g., 10 km thick)
- Sample at multiple offsets within the layer (e.g., 5 points)
- Average μ values across all samples in each layer
- Compute Δμ from the layer-averaged values

In [ ]:
# Layer-based sampling parameters
# Sample a thicker layer with more points to reduce noise from local heterogeneities
# layer_center_distance = 7500.0     # meters (7.5 km from fault to layer center)
layer_center_distance = 10000.0     # meters (7.5 km from fault to layer center)
layer_thickness = 15000.0          # meters (15 km thick layer)
n_samples_per_layer = 20           # Number of sampling points within each layer (for interpolation)

print(f"Subdomain IDs: blockleft={blockleft} (FW), blockright={blockright} (HW)")
print(f"\nLayer-based sampling configuration:")
print(f"  Layer center: {layer_center_distance/1e3:.1f} km from fault")
print(f"  Layer thickness: {layer_thickness/1e3:.1f} km")
print(f"  Sampling range: [{(layer_center_distance - layer_thickness/2)/1e3:.1f}, "
      f"{(layer_center_distance + layer_thickness/2)/1e3:.1f}] km from fault")

print("\n" + "="*70)
print("COMPUTING BOTH METHODS FOR COMPARISON")
print("="*70)

# # Diagnostic: Check mesh point distribution around fault
# print("\n" + "="*70)
# print("MESH POINT DISTRIBUTION ANALYSIS")
# print("="*70)

# from scipy.spatial import cKDTree

# # Build tree for mesh points
# tree_mesh = cKDTree(mu_3d_grid.points)

# # Sample a few fault points to check
# test_indices = np.linspace(0, len(fault_coords)-1, 20, dtype=int)

# print("\nChecking mesh point density along normals from fault:")
# print(f"{'Fault idx':<10} {'Side':<4} {'Distance (km)':<15} {'Nearest mesh (m)':<20} {'# points <5km'}")
# print("-" * 80)

# for idx in test_indices:
#     coord = fault_coords[idx]
#     normal = fault_normals[idx]

#     # Test at different distances along normal
#     for dist_km in [5, 10, 15, 20, 25]:
#         dist_m = dist_km * 1e3

#         # HW side
#         point_hw = coord + dist_m * normal
#         nearest_dist_hw, _ = tree_mesh.query(point_hw, k=1)
#         n_within_5km_hw = len(tree_mesh.query_ball_point(point_hw, r=5000))

#         # FW side  
#         point_fw = coord - dist_m * normal
#         nearest_dist_fw, _ = tree_mesh.query(point_fw, k=1)
#         n_within_5km_fw = len(tree_mesh.query_ball_point(point_fw, r=5000))

#         print(f"{idx:<10} {'HW':<4} {dist_km:<15} {nearest_dist_hw:<20.1f} {n_within_5km_hw}")
#         print(f"{idx:<10} {'FW':<4} {dist_km:<15} {nearest_dist_fw:<20.1f} {n_within_5km_fw}")

# # Check overall mesh statistics
# print("\n" + "="*70)
# print("MESH STATISTICS")
# print("="*70)

# # Compute approximate mesh spacing by finding nearest neighbor distances
# sample_pts = mu_3d_grid.points[::100]  # Sample every 100th point
# distances_to_neighbors = []
# tree_sample = cKDTree(mu_3d_grid.points)

# for pt in sample_pts[:1000]:  # Check 1000 points
#     dists, _ = tree_sample.query(pt, k=2)  # k=2 to get nearest neighbor (excluding itself)
#     distances_to_neighbors.append(dists[1])

# distances_to_neighbors = np.array(distances_to_neighbors)

# print(f"Mesh element size statistics (from nearest neighbor distances):")
# print(f"  Min:    {distances_to_neighbors.min()/1e3:.2f} km")
# print(f"  Median: {np.median(distances_to_neighbors)/1e3:.2f} km")
# print(f"  Mean:   {distances_to_neighbors.mean()/1e3:.2f} km")
# print(f"  Max:    {distances_to_neighbors.max()/1e3:.2f} km")
# print(f"  95th percentile: {np.percentile(distances_to_neighbors, 95)/1e3:.2f} km")

# print(f"\nTotal mesh points: {len(mu_3d_grid.points)}")
# print(f"Total fault points: {len(fault_coords)}")

# # Check if mesh spacing is larger than search distance at far distances
# if np.median(distances_to_neighbors) > 5000:
#     print(f"\nWARNING: Median mesh spacing ({np.median(distances_to_neighbors)/1e3:.2f} km) > search radius (5 km)")
#     print("This explains why many samples are returning NaN!")
#     print(f"Recommended: Increase max_search_dist to at least {np.percentile(distances_to_neighbors, 95)/1e3:.1f} km")


In [ ]:
# # METHOD 1: Linear interpolation (smooth, artificial)
# print("\n### METHOD 1: Linear Interpolation ###")
# mu_hw_interp, mu_fw_interp = sample_mu_across_fault_layered(
#     mu_3d_grid, fault_coords, fault_normals,
#     layer_center_distance, layer_thickness, n_samples_per_layer,
#     method='linear'
# )
# delta_mu_interp = mu_hw_interp - mu_fw_interp

# METHOD 2: Actual mesh vertices (true FEniCS values)
print("\n### METHOD 2: Actual Mesh Vertices ###")
mu_hw_mesh, mu_fw_mesh = sample_mu_across_fault_layered(
    mu_3d_grid, fault_coords, fault_normals,
    layer_center_distance, layer_thickness, n_samples_per_layer,
    field_name='shear modulus', 
    method='mesh_vertices'
)

delta_mu_mesh = mu_hw_mesh - mu_fw_mesh

print("\n" + "="*70)
print("COMPARISON OF METHODS")
print("="*70)

# print(f"\nΔμ (Linear Interpolation):")
# print(f"  Mean: {np.nanmean(delta_mu_interp):.2f} GPa")
# print(f"  Std:  {np.nanstd(delta_mu_interp):.2f} GPa")
# print(f"  Range: [{np.nanmin(delta_mu_interp):.2f}, {np.nanmax(delta_mu_interp):.2f}] GPa")

print(f"\nΔμ (Mesh Vertices - True FEniCS):")
print(f"  Mean: {np.nanmean(delta_mu_mesh):.2f} GPa")
print(f"  Std:  {np.nanstd(delta_mu_mesh):.2f} GPa")
print(f"  Range: [{np.nanmin(delta_mu_mesh):.2f}, {np.nanmax(delta_mu_mesh):.2f}] GPa")

# print(f"\nDifference between methods:")
# diff_methods = delta_mu_interp - delta_mu_mesh
# print(f"  Mean difference: {np.nanmean(diff_methods):.2f} GPa")
# print(f"  RMS difference:  {np.sqrt(np.nanmean(diff_methods**2)):.2f} GPa")


In [ ]:
### set the default 

# Default to mesh vertices method (true FEniCS values) for main analysis
mu_hw = mu_hw_mesh.copy()
mu_fw = mu_fw_mesh.copy()
delta_mu = delta_mu_mesh.copy()

# print(f"\n>>> Using MESH VERTICES method as default (delta_mu)")
# print(f">>> Interpolation results saved as delta_mu_interp for comparison")

# # Default to interpolated method for smoother results
# mu_hw = mu_hw_interp.copy()
# mu_fw = mu_fw_interp.copy()
# delta_mu = delta_mu_interp.copy()

In [ ]:
# # Comparison plot: Mesh vertices vs Linear interpolation (with optional smoothing)
# def plot_method_comparison(fault_coords, delta_mu_mesh, delta_mu_interp, smooth=False, sigma=2.0):
#     """Compare mesh vertices and linear interpolation methods"""
#     fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
#     if smooth:
#         # Apply smoothing for visualization
#         coords_smooth_mesh, delta_mu_mesh_plot = smooth_fault_data(fault_coords, delta_mu_mesh, sigma=sigma)
#         coords_smooth_interp, delta_mu_interp_plot = smooth_fault_data(fault_coords, delta_mu_interp, sigma=sigma)
#         x_km_mesh = coords_smooth_mesh[:, 0] / 1e3
#         y_km_mesh = coords_smooth_mesh[:, 1] / 1e3
#         x_km_interp = coords_smooth_interp[:, 0] / 1e3
#         y_km_interp = coords_smooth_interp[:, 1] / 1e3
#         title_suffix = f' (smoothed, σ={sigma})'
#     else:
#         x_km_mesh = fault_coords[:, 0] / 1e3
#         y_km_mesh = fault_coords[:, 1] / 1e3
#         x_km_interp = x_km_mesh
#         y_km_interp = y_km_mesh
#         delta_mu_mesh_plot = delta_mu_mesh
#         delta_mu_interp_plot = delta_mu_interp
#         title_suffix = ''
    
#     # Find common color scale
#     vmax = max(np.nanmax(np.abs(delta_mu_mesh_plot)), np.nanmax(np.abs(delta_mu_interp_plot)))
    
#     # Panel 1: Mesh vertices (true FEniCS)
#     sc1 = axes[0].scatter(x_km_mesh, y_km_mesh, c=delta_mu_mesh_plot, s=50,
#                           cmap='RdBu_r', vmin=-vmax, vmax=vmax, alpha=0.8)
#     axes[0].set_xlabel('X (km)')
#     axes[0].set_ylabel('Y (km)')
#     axes[0].set_title('Δμ (Mesh Vertices [True FEniCS])' + title_suffix, fontsize=12, fontweight='bold')
#     axes[0].set_aspect('equal')
#     axes[0].grid(True, alpha=0.3)
#     cbar1 = plt.colorbar(sc1, ax=axes[0])
#     cbar1.set_label('Δμ (GPa)')
#     axes[0].text(0.02, 0.98, f'Mean = {np.nanmean(delta_mu_mesh):.2f} GPa\nStd = {np.nanstd(delta_mu_mesh):.2f} GPa',
#                  transform=axes[0].transAxes, fontsize=9, va='top',
#                  bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
#     # Panel 2: Linear interpolation
#     sc2 = axes[1].scatter(x_km_interp, y_km_interp, c=delta_mu_interp_plot, s=50,
#                           cmap='RdBu_r', vmin=-vmax, vmax=vmax, alpha=0.8)
#     axes[1].set_xlabel('X (km)')
#     axes[1].set_ylabel('Y (km)')
#     axes[1].set_title('Δμ (Linear Interpolation [Smoothed])' + title_suffix, fontsize=12, fontweight='bold')
#     axes[1].set_aspect('equal')
#     axes[1].grid(True, alpha=0.3)
#     cbar2 = plt.colorbar(sc2, ax=axes[1])
#     cbar2.set_label('Δμ (GPa)')
#     axes[1].text(0.02, 0.98, f'Mean = {np.nanmean(delta_mu_interp):.2f} GPa\nStd = {np.nanstd(delta_mu_interp):.2f} GPa',
#                  transform=axes[1].transAxes, fontsize=9, va='top',
#                  bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
#     # Panel 3: Difference
#     diff = delta_mu_mesh - delta_mu_interp
#     if smooth:
#         coords_smooth_diff, diff_plot = smooth_fault_data(fault_coords, diff, sigma=sigma)
#         x_km_diff = coords_smooth_diff[:, 0] / 1e3
#         y_km_diff = coords_smooth_diff[:, 1] / 1e3
#     else:
#         x_km_diff = x_km_mesh
#         y_km_diff = y_km_mesh
#         diff_plot = diff
    
#     vmax_diff = np.nanmax(np.abs(diff_plot))
#     sc3 = axes[2].scatter(x_km_diff, y_km_diff, c=diff_plot, s=50,
#                           cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff, alpha=0.8)
#     axes[2].set_xlabel('X (km)')
#     axes[2].set_ylabel('Y (km)')
#     axes[2].set_title('Difference (Mesh - Interp)' + title_suffix, fontsize=12, fontweight='bold')
#     axes[2].set_aspect('equal')
#     axes[2].grid(True, alpha=0.3)
#     cbar3 = plt.colorbar(sc3, ax=axes[2])
#     cbar3.set_label('Δμ difference (GPa)')
#     axes[2].text(0.02, 0.98, f'RMS diff = {np.sqrt(np.nanmean(diff**2)):.2f} GPa',
#                  transform=axes[2].transAxes, fontsize=9, va='top',
#                  bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
#     plt.tight_layout()
#     return fig

# # Plot without smoothing
# print("Method comparison WITHOUT smoothing:")
# fig1 = plot_method_comparison(fault_coords, delta_mu_mesh, delta_mu_interp, smooth=False)
# plt.savefig(resultpath + 'delta_mu_method_comparison_raw.png', dpi=300, bbox_inches='tight')
# plt.show()

# # Plot with smoothing
# print("\nMethod comparison WITH smoothing (σ=2):")
# fig2 = plot_method_comparison(fault_coords, delta_mu_mesh, delta_mu_interp, smooth=True, sigma=2.0)
# plt.savefig(resultpath + 'delta_mu_method_comparison_smooth.png', dpi=300, bbox_inches='tight')
# plt.show()

# print(f"\nMethod comparison figures saved to: {resultpath}delta_mu_method_comparison_*.png")
# print(f"\nKey observation:")
# print(f"  - Mesh vertices: Shows true discretization structure from FEniCS")
# print(f"  - Linear interp: Artificially smoothed during sampling")
# print(f"  - Visualization smoothing: Applied ONLY for plotting, data unchanged")
# print(f"  - RMS difference: {np.sqrt(np.nanmean((delta_mu_mesh - delta_mu_interp)**2)):.2f} GPa")

In [ ]:
# # Plot Δμ on fault surface with optional smoothing
# def plot_delta_mu(fault_coords, delta_mu, title='Δμ Across Fault (μ_HW - μ_FW)', 
#                   smooth=False, sigma=2.0, save_suffix=''):
#     """Plot delta mu with optional smoothing"""
#     fig, ax = plt.subplots(figsize=(10, 6))
    
#     if smooth:
#         coords_smooth, delta_mu_smooth = smooth_fault_data(fault_coords, delta_mu, sigma=sigma)
#         x_km = coords_smooth[:, 0] / 1e3
#         y_km = coords_smooth[:, 1] / 1e3
#         plot_values = delta_mu_smooth
#         title = title + f' (smoothed, σ={sigma})'
#     else:
#         x_km = fault_coords[:, 0] / 1e3
#         y_km = fault_coords[:, 1] / 1e3
#         plot_values = delta_mu
    
#     # Plot with diverging colormap
#     vmax = np.nanmax(np.abs(plot_values))
#     sc = ax.scatter(x_km, y_km, c=plot_values, s=50,
#                     cmap='RdBu_r', vmin=-vmax, vmax=vmax, alpha=0.8)
    
#     ax.set_xlabel('X (km)', fontsize=12)
#     ax.set_ylabel('Y (km)', fontsize=12)
#     ax.set_title(title, fontsize=14, fontweight='bold')
#     ax.set_aspect('equal')
#     ax.grid(True, alpha=0.3)
    
#     cbar = plt.colorbar(sc, ax=ax)
#     cbar.set_label('Δμ (GPa)', fontsize=12)
    
#     # Add text annotations
#     ax.text(0.02, 0.98, f'Mean Δμ = {np.nanmean(delta_mu):.2f} GPa',
#             transform=ax.transAxes, fontsize=10, va='top',
#             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
#     plt.tight_layout()
#     if flag_savefig:
#         plt.savefig(resultpath + f'delta_mu_fault{save_suffix}.png', dpi=300, bbox_inches='tight')
#     return fig

# # Plot without smoothing
# print("Plotting Δμ without smoothing:")
# fig1 = plot_delta_mu(fault_coords, delta_mu, smooth=False, save_suffix='_raw')
# plt.show()

# # Plot with moderate smoothing
# print("\nPlotting Δμ with smoothing (σ=2):")
# fig2 = plot_delta_mu(fault_coords, delta_mu, smooth=True, sigma=2.0, save_suffix='_smooth_s2')
# plt.show()

# # Plot with stronger smoothing
# print("\nPlotting Δμ with smoothing (σ=3):")
# fig3 = plot_delta_mu(fault_coords, delta_mu, smooth=True, sigma=3.0, save_suffix='_smooth_s3')
# plt.show()

# print(f"\nFigures saved to: {resultpath}delta_mu_fault_*.png")

In [ ]:
# # 8-panel PyGMT figure: μ_FW, μ_fault_3d, μ_HW and their anomalies.
# lon_stack = np.concatenate([
#     np.asarray(fault_lon, dtype=float),
#     m_s_3d['lon'].values.astype(float),
#     df_obs['lon'].values.astype(float)
# ])
# lat_stack = np.concatenate([
#     np.asarray(fault_lat, dtype=float),
#     m_s_3d['lat'].values.astype(float),
#     df_obs['lat'].values.astype(float)
# ])
# pad_deg = 0.25
# region_fault = [
#     float(lon_stack.min() - pad_deg),
#     float(lon_stack.max() + pad_deg),
#     float(lat_stack.min() - pad_deg),
#     float(lat_stack.max() + pad_deg)
# ]

# plate_grd = pygmt.xyz2grd(
#     x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'],
#     region=region_fault, spacing=(0.05, 0.05)
# )

# style_fault = "c0.11c"
# colorbar_params = "+o0/-0.75c+w5.8c/0.32c+h"

# mu_all = np.concatenate([mu_fw, mu_fault_3d, mu_hw]).astype(float)
# mu_min = float(np.nanmin(mu_all))
# mu_max = float(np.nanmax(mu_all))
# mu_step = max((mu_max - mu_min) / 20.0, 1e-3)
# print(f"μ range: [{mu_min:.3f}, {mu_max:.3f}] GPa, step: {mu_step:.4f} GPa")

# mu_fw_anom = (mu_fw - mu_ref) / mu_ref * 100.0
# mu_fault_anom = (mu_fault_3d - mu_ref) / mu_ref * 100.0
# mu_hw_anom = (mu_hw - mu_ref) / mu_ref * 100.0
# anom_all = np.concatenate([mu_fw_anom, mu_fault_anom, mu_hw_anom]).astype(float)
# anom_vmax = max(float(np.nanmax(np.abs(anom_all))), 1e-2)
# anom_step = max(anom_vmax / 10.0, 1e-3)
# print(f"Anomaly range: ±{anom_vmax:.3f} %, step: {anom_step:.4f} %")

# slip_diff_values = m_s_3d['slip_diff'].values
# slip_vmax = max(float(np.nanmax(np.abs(slip_diff_values))), 1e-4)
# slip_step = max(slip_vmax / 10.0, 1e-5)
# print(f"Slip/Coupling diff range: ±{slip_vmax:.5f}, step: {slip_step:.6f}")

# delta_mu_rel_hom = (mu_hw - mu_fw) / mu_ref * 100.0
# delta_rel_vmax = max(float(np.nanmax(np.abs(delta_mu_rel_hom))), 1e-2)
# delta_rel_step = max(delta_rel_vmax / 10.0, 1e-3)
# print(f"Δμ relative to hom range: ±{delta_rel_vmax:.3f} %, step: {delta_rel_step:.4f} %")   

# fig = pygmt.Figure()

# def _panel_basemap(title):
#     fig.basemap(region=region_fault, projection="M?", frame=["a1f0.2", f"+t{title}"])

# def _panel_overlays(fill_land=False):
#     coast_kwargs = dict(
#         shorelines="0.25p,black",
#         area_thresh=20,
#         land="lightgray" if fill_land else None,
#         water=None,
#         resolution="h"
#     )
#     fig.coast(**coast_kwargs)
#     fig.grdcontour(grid=plate_grd, levels=5, limit="-100/-5",
#                    annotation="20+f6p", pen="0.4p,darkgray")
#     fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray")

# with fig.subplot(nrows=2, ncols=4, figsize=("27c", "15c"), autolabel=False,
#                  sharex=True, sharey=True, margins="0.45c/0.6c"):
#     pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
#                  MAP_TITLE_OFFSET="0.3c", FONT_ANNOT_PRIMARY="7p")

#     mu_titles = ["(a) μ_FW (Layer Avg)", "(b) μ on Fault (3D)",
#                  "(c) μ_HW (Layer Avg)", "(d) Δ Coupling (3D - Hom)"]
#     mu_fields = [mu_fw, mu_fault_3d, mu_hw, slip_diff_values]
#     for idx, (title, data) in enumerate(zip(mu_titles, mu_fields)):
#         with fig.set_panel(panel=[0, idx]):
#             _panel_basemap(title)
#             if idx < 3:
#                 pygmt.makecpt(cmap="rainbow", series=[mu_min, mu_max, mu_step], background=True)
#                 fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=data, cmap=True)
#                 label = "μ (GPa)"
#             else:
#                 pygmt.makecpt(cmap="roma", series=[-slip_vmax, slip_vmax, slip_step], background=True)
#                 fig.plot(x=m_s_3d['lon'], y=m_s_3d['lat'], style=style_fault, fill=data, cmap=True)
#                 label = "Δ Coupling"
#             _panel_overlays(fill_land=False)
#             with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
#                 fig.colorbar(frame=["af", f"x+l{label}"])

#     anom_titles = ["(e) μ_FW Anomaly (%)", "(f) μ Fault Anomaly (%)",
#                    "(g) μ_HW Anomaly (%)", "(h) Δμ Relative to Hom (%)"]
#     anom_fields = [mu_fw_anom, mu_fault_anom, mu_hw_anom, delta_mu_rel_hom]
#     for idx, (title, data) in enumerate(zip(anom_titles, anom_fields)):
#         with fig.set_panel(panel=[1, idx]):
#             _panel_basemap(title)
#             if idx < 3:
#                 pygmt.makecpt(cmap="roma", series=[-anom_vmax, anom_vmax, anom_step], background=True)
#                 fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=data, cmap=True)
#                 label = "Anomaly (%)"
#             else:
#                 pygmt.makecpt(cmap="roma", series=[-delta_rel_vmax, delta_rel_vmax, delta_rel_step], background=True)
#                 fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=data, cmap=True)
#                 label = "Δμ (%)"
#             _panel_overlays(fill_land=False)
#             with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
#                 fig.colorbar(frame=["af", f"x+l{label}"])   #position=f"JCB{colorbar_params}", 

# fig.show()
# if flag_savefig:
#     fig.savefig(resultpath + 'mu_layer_vs_fault_8panel.png', dpi=300)
# print(f"Layer-resolved μ figure saved to: {resultpath}mu_layer_vs_fault_8panel.png")



In [ ]:
# # 4-panel PyGMT figure: μ_fault_3d, μ_HW-μ_FW difference and their anomalies/deltas
# lon_stack = np.concatenate([
#     np.asarray(fault_lon, dtype=float),
#     m_s_3d['lon'].values.astype(float),
#     df_obs['lon'].values.astype(float)
# ])
# lat_stack = np.concatenate([
#     np.asarray(fault_lat, dtype=float),
#     m_s_3d['lat'].values.astype(float),
#     df_obs['lat'].values.astype(float)
# ])
# pad_deg = 0.25
# region_fault = [
#     float(lon_stack.min() - pad_deg),
#     float(lon_stack.max() + pad_deg),
#     float(lat_stack.min() - pad_deg),
#     float(lat_stack.max() + pad_deg)
# ]
# region_fault = region

# plate_grd = pygmt.xyz2grd(
#     x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'],
#     region=region_fault, spacing=(0.05, 0.05)
# )

# style_fault = "c0.1c"
# colorbar_params = "+o0/-0.75c+w5.8c/0.32c+h"

# mu_all = np.concatenate([mu_fw, mu_fault_3d, mu_hw]).astype(float)
# mu_min = float(np.nanmin(mu_all))
# mu_max = float(np.nanmax(mu_all))
# mu_step = max((mu_max - mu_min) / 20.0, 1e-3)
# print(f"μ range: [{mu_min:.3f}, {mu_max:.3f}] GPa, step: {mu_step:.4f} GPa")

# mu_fw_anom = (mu_fw - mu_ref) / mu_ref * 100.0
# mu_fault_anom = (mu_fault_3d - mu_ref) / mu_ref * 100.0
# mu_hw_anom = (mu_hw - mu_ref) / mu_ref * 100.0
# anom_all = np.concatenate([mu_fw_anom, mu_fault_anom, mu_hw_anom]).astype(float)
# anom_vmax = max(float(np.nanmax(np.abs(anom_all))), 1e-2)
# anom_step = max(anom_vmax / 10.0, 1e-3)
# print(f"Anomaly range: ±{anom_vmax:.3f} %, step: {anom_step:.4f} %")

# slip_diff_values = m_s_3d['slip_diff'].values
# slip_vmax = max(float(np.nanmax(np.abs(slip_diff_values))), 1e-4)
# slip_step = max(slip_vmax / 10.0, 1e-5)
# print(f"Slip/Coupling diff range: ±{slip_vmax:.5f}, step: {slip_step:.6f}")

# # Calculate absolute difference μ_HW - μ_FW
# mu_hw_minus_fw = mu_hw - mu_fw
# mu_diff_vmax = max(float(np.nanmax(np.abs(mu_hw_minus_fw))), 1e-3)
# mu_diff_step = max(mu_diff_vmax / 10.0, 1e-4)
# print(f"μ_HW - μ_FW range: ±{mu_diff_vmax:.3f} GPa, step: {mu_diff_step:.4f} GPa")

# delta_mu_rel_hom = (mu_hw - mu_fw) / mu_ref * 100.0
# delta_rel_vmax = max(float(np.nanmax(np.abs(delta_mu_rel_hom))), 1e-2)
# delta_rel_step = max(delta_rel_vmax / 10.0, 1e-3)
# print(f"Δμ relative to hom range: ±{delta_rel_vmax:.3f} %, step: {delta_rel_step:.4f} %")


# fig = pygmt.Figure()

# def _panel_basemap(title):
#     fig.basemap(region=region_fault, projection="M?", frame=["a1f0.2", f"+t{title}"])

# def _panel_overlays(fill_land=False):
#     coast_kwargs = dict(
#         shorelines="0.25p,black",
#         area_thresh=20,
#         land="lightgray" if fill_land else None,
#         water=None,
#         resolution="h"
#     )
#     fig.coast(**coast_kwargs)
#     fig.grdcontour(grid=plate_grd, levels=5, limit="-100/-5",
#                     annotation="20+f6p", pen="0.4p,darkgray")
#     fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray")

# with fig.subplot(nrows=2, ncols=2, figsize=("12c", "14c"), autolabel=False,
#                 sharex=True, sharey=True, margins="0.45c/0.6c"):
#     pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
#                 MAP_TITLE_OFFSET="0.3c", FONT_ANNOT_PRIMARY="7p")

#     # Row 1: Panel (b) and Panel with μ_HW - μ_FW
#     # Panel (b): μ on Fault (3D)
#     with fig.set_panel(panel=[0, 0]):
#         _panel_basemap("(a) μ on Fault (3D)")
#         pygmt.makecpt(cmap="rainbow", series=[mu_min, mu_max, mu_step], background=True)
#         fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=mu_fault_3d,
# cmap=True)
#         _panel_overlays(fill_land=False)
#         with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
#             fig.colorbar(frame=["af", "x+lμ (GPa)"])

#     # NEW Panel: μ_HW - μ_FW (absolute difference)
#     with fig.set_panel(panel=[0, 1]):
#         _panel_basemap("(b) μ_HW - μ_FW")
#         pygmt.makecpt(cmap="roma", series=[-mu_diff_vmax, mu_diff_vmax, mu_diff_step],
# background=True)
#         fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=mu_hw_minus_fw,
# cmap=True)
#         _panel_overlays(fill_land=False)
#         with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
#             fig.colorbar(frame=["af", "x+lΔμ (GPa)"])

#     # Row 2: Panel (f) and Panel (h)
#     # Panel (f): μ Fault Anomaly (%)
#     with fig.set_panel(panel=[1, 0]):
#         _panel_basemap("(c) μ Fault Anomaly (%)")
#         pygmt.makecpt(cmap="roma", series=[-anom_vmax, anom_vmax, anom_step],
# background=True)
#         fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=mu_fault_anom,
# cmap=True)
#         _panel_overlays(fill_land=False)
#         with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
#             fig.colorbar(frame=["af", "x+lAnomaly (%)"])

#     # Panel (h): Δμ Relative to Hom (%)
#     with fig.set_panel(panel=[1, 1]):
#         _panel_basemap("(d) Δμ Relative to Hom (%)")
#         pygmt.makecpt(cmap="roma", series=[-delta_rel_vmax, delta_rel_vmax,
# delta_rel_step], background=True)
#         fig.plot(x=fault_lon, y=fault_lat, style=style_fault, fill=delta_mu_rel_hom,
# cmap=True)
#         _panel_overlays(fill_land=False)
#         with pygmt.config(FONT_ANNOT_PRIMARY="10p"):
#             fig.colorbar(frame=["af", "x+lΔμ (%)"])

# fig.show()
# if flag_savefig:
#     fig.savefig(resultpath + 'mu_fault_hw_4panel.png', dpi=300)
# print(f"4-panel figure saved to: {resultpath}mu_fault_hw_4panel.png")

## 10.5. Horizontal Slices of μ Structure

Visualize how μ and μ anomaly vary with depth using horizontal slices from 0 to 80 km depth.

**Purpose:**
- Understand depth variation of shear modulus structure
- Investigate how μ contrasts change with depth
- Relate depth-dependent structure to slip inversion bias

**Implementation:**
Two approaches for creating horizontal slices:
1. **Direct griddata interpolation**: Fast but can be grainy due to mesh discretization
2. **PyVista slicing** (recommended): Uses PyVista's native slicing for smoother results

**Visualization:**
- 3×3 panel layout showing 9 depth levels (0, -10, -20, ..., -80 km)
- PyGMT style with coastlines, plate interface contours, and trench
- Consistent colormaps: rainbow for μ, roma for anomaly

In [ ]:
# IMPROVED VERSION: Horizontal slices using PyVista slicing + PyGMT surface
# This approach: Irregular mesh -> PyVista slice -> blockmedian -> pygmt.surface -> smooth grid
# Avoids double interpolation (griddata + xyz2grd) and eliminates GMT warnings

import numpy as np
import pygmt
import pyvista as pv

# ============================================================================
# SECTION 1: Create horizontal slices (modified to keep scattered points)
# ============================================================================

# ### 0-80 km. 9 depths
# depth_levels = np.arange(0.0, -80.1, -10.0)  # 0 to -80 km, 9 depths
# region_fault = region  # Assuming this is defined

### 20-50 km. 4 depths
depth_levels = np.arange(-20.0, -50.1, -10.0)  # -20 to -50 km, 4 depths
region_fault = [-86.5, -84.5, 9, 11]

# ### 20-50 km. 4 depths
# depth_levels = np.arange(-25.0, -55.1, -10.0)  # -20 to -50 km, 4 depths
# region_fault = [-86.5, -84.5, 9, 11]

# Define vertical slice positions
x_strike_km = np.arange(-50, 51, 20)  # Along-strike slices (constant x)
y_min, y_max = -100, 100  # km

# y_dip_km = np.arange(-50, 51, 20)     # Along-dip slices (constant y)
y_dip_km = np.arange(-10, 51, 20)  # [-30, -10, 10, 30] - 4 slices
# x_min, x_max = -80, 120   # km
x_min, x_max = -100, 100   # km

# Storage for scattered point data (NOT gridded!)
mu_slice_scattered = []      # List of dicts with {lon, lat, mu}

print("Creating horizontal slices using PyVista slicing...")
for i, depth_km in enumerate(depth_levels):
    z_value = depth_km * 1e3  # Convert to meters

    # Use PyVista's native slicing
    normal = [0., 0., 1.]  # Horizontal slice
    origin = [0., 0., z_value]

    # Slice the 3D grid
    slice_3d = mu_3d_grid.slice(normal=normal, origin=origin)

    if len(slice_3d.points) == 0:
        print(f"  Depth {depth_km:.0f} km: No data")
        mu_slice_scattered.append(None)
        continue

    # Get sliced points and values (KEEP AS SCATTERED DATA)
    slice_points = slice_3d.points
    slice_mu = slice_3d.point_data['shear modulus']

    # Convert local coordinates to lon/lat
    x_pts = slice_points[:, 0] + x0
    y_pts = slice_points[:, 1] + y0
    lon_pts, lat_pts = utp.ckm2LLd(x_pts, y_pts, lon0, lat0, -rot)

    # Filter points to only keep those within region_fault
    mask = (
        (lon_pts >= region_fault[0]) & (lon_pts <= region_fault[1]) &
        (lat_pts >= region_fault[2]) & (lat_pts <= region_fault[3])
    )

    lon_pts_filtered = lon_pts[mask]
    lat_pts_filtered = lat_pts[mask]
    mu_filtered = slice_mu[mask]
    
    if len(lon_pts_filtered) == 0:
        print(f"  Depth {depth_km:.0f} km: No data within region_fault")
        mu_slice_scattered.append(None)
        continue

    # Display mu statistics for this depth slice
    mu_min = mu_filtered.min()
    mu_max = mu_filtered.max()
    mu_mean = np.mean(mu_filtered)
    mu_std = np.std(mu_filtered)

    print(f"    μ range: [{mu_min:.2f}, {mu_max:.2f}] GPa, "
        f"mean: {mu_mean:.2f} GPa, std: {mu_std:.2f} GPa")

    # Store as scattered points (NO griddata here!)
    mu_slice_scattered.append({
        'lon': lon_pts_filtered,
        'lat': lat_pts_filtered,
        'mu': mu_filtered,
        'depth_km': depth_km
    })

    print(f"  Depth {depth_km:.0f} km: {len(lon_pts)} total → {len(lon_pts_filtered)} within region_fault")

print(f"\nCreated {len([s for s in mu_slice_scattered if s is not None])} slices")


# ============================================================================
# SECTION 2: Calculate reference mu and anomaly slices
# ============================================================================

# Collect all mu values from all slices to get global min/max
all_mu_values = []
for slice_data in mu_slice_scattered:
    if slice_data is not None:
        all_mu_values.extend(slice_data['mu'])

all_mu_values = np.array(all_mu_values)
mu_global_min = all_mu_values.min()
mu_global_max = all_mu_values.max()

# Calculate reference mu (arithmetic mean of global min and max)
# mu_ref = (mu_global_min + mu_global_max) / 2.0

print(f"\nGlobal mu statistics:")
print(f"  Min: {mu_global_min:.2f} GPa")
print(f"  Max: {mu_global_max:.2f} GPa")
# print(f"  mu_ref (mean of min/max): {mu_ref:.2f} GPa")

# # Create anomaly slices relative to mu_ref
# anom_slice_scattered = []
# for slice_data in mu_slice_scattered:
#     if slice_data is None:
#         anom_slice_scattered.append(None)
#     else:
#         anom = (slice_data['mu'] - mu_ref) / mu_ref * 100  # Percentage
#         anom_slice_scattered.append({
#             'lon': slice_data['lon'],
#             'lat': slice_data['lat'],
#             'anom': anom,
#             'depth_km': slice_data['depth_km']
#         })

# print(f"Created anomaly slices relative to mu_ref = {mu_ref:.2f} GPa")


# ============================================================================
# SECTION 3: Plotting function using blockmedian + pygmt.surface
# ============================================================================

def _plot_slice_panels_flex_surface(data_slices_scattered, depths_km, cmap, series,
                                    label="Value",  # Keep as default/fallback
                                    colorbar_labels=None,  # NEW: individual labels per panel
                                    show_colorbar_per_panel=False,  # NEW: enable per-panel colorbars
                                    filename=None, nrows=3, ncols=3,
                                    spacing="0.03d", tension=0.25,
                                    use_blockmedian=True, block_spacing=None):
    """Plot horizontal slices in PyGMT style using surface interpolation

    Parameters
    ----------
    data_slices_scattered : list of dict
        List of dicts with keys: 'lon', 'lat', and data key (e.g., 'mu', 'anom')
        Each dict contains scattered point data for one depth slice
    depths_km : list
        Depths in km for each slice
    cmap : str
        PyGMT colormap name
    series : list
        [min, max, step] for colormap
    label : str
        Label for colorbar
    filename : str
        Output filename
    nrows : int
        Number of rows in subplot grid (default 3)
    ncols : int
        Number of columns in subplot grid (default 3)
    spacing : str
        Grid spacing for surface interpolation (default "0.03d" ≈ 3.3 km)
    tension : float
        Tension parameter for surface (0=smooth, 1=tight to data, default 0.25)
    use_blockmedian : bool
        Whether to preprocess with blockmedian (default True, recommended)
    block_spacing : str, optional
        Spacing for blockmedian. If None, uses same as spacing parameter
    """
    # Determine data key (mu, anom, etc.)
    data_key = None
    for slice_data in data_slices_scattered:
        if slice_data is not None:
            for key in slice_data.keys():
                if key not in ['lon', 'lat', 'depth_km']:
                    data_key = key
                    break
            break

    if data_key is None:
        raise ValueError("Could not determine data key in scattered slices")

    # Set block spacing
    if block_spacing is None:
        block_spacing = spacing

    # Calculate figure size based on grid dimensions
    fig_width = f"{ncols * 7}c"
    fig_height = f"{nrows * 8}c"

    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

    fig = pygmt.Figure()
    with fig.subplot(nrows=nrows, ncols=ncols, figsize=(fig_width, fig_height),
                    autolabel=False, sharex=True, sharey=True,
                    margins="0.1c/0.15c", frame=["WSne"]):
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
                    MAP_TITLE_OFFSET="-0.15c", FONT_ANNOT_PRIMARY="7p")

        for idx_panel, (depth_km, slice_data) in enumerate(zip(depths_km, data_slices_scattered)):
            row = idx_panel // ncols
            col = idx_panel % ncols

            with fig.set_panel(panel=[row, col]):
                fig.basemap(region=region_fault, projection="M?",
                            frame=["a1f0.2"])

                # Check if we have data for this slice
                if slice_data is None:
                    # No data - just show basemap
                    fig.coast(shorelines="0.5p,black", area_thresh=100)
                    fig.text(x=(region_fault[0] + region_fault[1])/2,
                            y=(region_fault[2] + region_fault[3])/2,
                            text="No data", font="12p,Helvetica-Bold,red",
                            justify="CM")
                else:
                    # Preprocess with blockmedian to remove duplicates and average nearby points
                    if use_blockmedian:
                        print(f"  Panel {idx_panel}: Original points: {len(slice_data['lon'])}", end='')
                        # Use blockmedian to average points in grid cells
                        filtered_data = pygmt.blockmedian(
                            x=slice_data['lon'],
                            y=slice_data['lat'],
                            z=slice_data[data_key],
                            region=region_fault,
                            spacing=block_spacing,
                            registration="gridline"  # Fix region/spacing compatibility
                        )
                        # blockmedian returns a pandas DataFrame with columns indexed by position
                        x_filtered = filtered_data.iloc[:, 0].values  # First column: x/lon
                        y_filtered = filtered_data.iloc[:, 1].values  # Second column: y/lat
                        z_filtered = filtered_data.iloc[:, 2].values  # Third column: z/data
                        print(f" → Filtered to: {len(x_filtered)} points")
                    else:
                        # Use original scattered points
                        x_filtered = slice_data['lon']
                        y_filtered = slice_data['lat']
                        z_filtered = slice_data[data_key]

                    # Use pygmt.surface on filtered points
                    grid = pygmt.surface(
                        x=x_filtered,
                        y=y_filtered,
                        z=z_filtered,
                        region=region_fault,
                        spacing=spacing,
                        tension=tension,
                        registration="gridline"  # Fix region/spacing compatibility
                    )

                    # Make colormap and plot
                    pygmt.makecpt(cmap=cmap, series=series, background=True)
                    fig.grdimage(grid=grid, cmap=True)

                    fig.coast(shorelines="0.5p,black", area_thresh=100)

                    # Add plate interface contours (if plate_grd is defined)
                    try:
                        fig.grdcontour(grid=plate_grd, levels=5, limit="-100/-5",
                                    annotation="20+f6p", pen="0.4p,darkgray")
                    except:
                        pass  # Skip if plate_grd not available

                    # Individual colorbar for each panel with its own mu_ref label
                    if colorbar_labels is not None and idx_panel < len(colorbar_labels):
                        panel_label = colorbar_labels[idx_panel]
                    else:
                        panel_label = label

                    # Colorbar - either per panel or only on right column
                    if show_colorbar_per_panel:

                        with pygmt.config(FONT_ANNOT_PRIMARY="9p"):
                            if len(series) < 3:
                                fig.colorbar(frame=["af", f"x+l{panel_label}"], position="JBC+h+o0/0.25c")
                            else:
                                fig.colorbar(frame=[f"a{series[2]*4}f{series[2]}", f"x+l{panel_label}"],
                                            position="JBC+h+o0/0.25c")
                    else:
                        # Original behavior: colorbar only on right column with shared label
                        # if col == ncols - 1:
                        if row == 0:
                            with pygmt.config(FONT_ANNOT_PRIMARY="9p"):
                                if len(series) < 3:
                                    fig.colorbar(frame=["af", f"x+l{panel_label}"], position="JBC+h+o0/0.25c")
                                else:   
                                    fig.colorbar(frame=[f"a{series[2]*4}f{series[2]}", f"x+l{panel_label}"],
                                            position="JBC+h+o0/0.25c")
                                    # fig.colorbar(frame=[f"a{30}f{15}", f"x+l{panel_label}"],
                                    #         position="JBC+h+o0/0.25c")
                                
                    # Plot vertical slice locations on first panel (if defined)
                    if idx_panel == 0:
                        try:
                            # # Along-strike slices (constant x, blue dashed)
                            # for x_km in x_strike_km:
                            #     x_m = x_km * 1e3
                            #     y_line = np.linspace(y_min * 1e3, y_max * 1e3, 5)
                            #     x_line = np.full_like(y_line, x_m)
                            #     lon_line, lat_line = utp.ckm2LLd(x_line + x0, y_line + y0,
                            #                                     lon0, lat0, -rot)
                            #     fig.plot(x=lon_line, y=lat_line, pen="0.8p,white,--")

                            # Along-dip slices (constant y, red dashed)
                            for y_km in y_dip_km:
                                y_m = y_km * 1e3
                                x_line = np.linspace(x_min * 1e3, x_max * 1e3, 5)
                                y_line = np.full_like(x_line, y_m)
                                lon_line, lat_line = utp.ckm2LLd(x_line + x0, y_line + y0,
                                                                lon0, lat0, -rot)
                                fig.plot(x=lon_line, y=lat_line, pen="0.8p,white,--")
                        except:
                            pass  # Skip if slice locations not defined

                    # Add depth label
                    fig.text(x=region_fault[1] - 0.05, y=region_fault[3] - 0.05,
                            text=f"Depth={abs(depth_km):.0f} km",
                            font="10p,Helvetica-Bold,white", justify="TR",
                            fill="gray60", offset="0.1c/0.1c",)   # fill="black@50", pen="0.5p,white"

                    #plot panel label
                    panel_label = panel_labels[idx_panel]
                    fig.text(text=panel_label, x=region_fault[0]+0.1, y=region_fault[-1]-0.08, 
                             font="11p,Helvetica-Bold,black", justify="CM")

    fig.show()
    if flag_savefig:
        # fig.savefig(filename, dpi=300, transparent=False)   #
        fig.savefig(filename.replace('.pdf', '.png'), dpi=600)

    print(f"Saved slice figure to: {filename}")


# ============================================================================
# SECTION 4: Alternative - Use custom mu_ref for anomaly calculation
# ============================================================================

def calculate_anomaly_slices(mu_slices, mu_ref=None, use_slice_mean=False):
    """Calculate anomaly slices relative to a reference mu value
    
    Parameters
    ----------
    mu_slices : list of dict
        List of mu slice data (from Section 1)
    mu_ref : float, optional
        Reference mu value for anomaly calculation
        If None and use_slice_mean=False, uses arithmetic mean of global min/max (default)
        Ignored if use_slice_mean=True
    use_slice_mean : bool, optional
        If True, each slice uses its own mean as mu_ref (default: False)
        This option overrides the mu_ref parameter
    
    Returns
    -------
    anom_slices : list of dict
        List of anomaly slice data
    mu_ref : float or list of float
        The reference value(s) used. If use_slice_mean=True, returns list of mu_ref per slice
    """

    if use_slice_mean:
        print("Using each slice's own mean as mu_ref for anomaly calculation")
        anom_slices = []
        mu_ref_list = []

        for slice_data in mu_slices:
            if slice_data is None:
                anom_slices.append(None)
                mu_ref_list.append(None)
            else:
                # Use this slice's mean as its reference
                slice_mu_ref = np.mean(slice_data['mu'])
                mu_ref_list.append(slice_mu_ref)

                anom = (slice_data['mu'] - slice_mu_ref) / slice_mu_ref * 100  # Percentage
                anom_slices.append({
                    'lon': slice_data['lon'],
                    'lat': slice_data['lat'],
                    'anom': anom,
                    'depth_km': slice_data['depth_km'],
                    'mu_ref': slice_mu_ref  # Store the reference used for this slice
                })
                print(f"  Depth {slice_data['depth_km']:.0f} km: mu_ref = {slice_mu_ref:.2f} GPa")

        return anom_slices, mu_ref_list

    else:
        # Original behavior: use global or custom mu_ref
        # Collect all mu values if mu_ref not provided
        if mu_ref is None:
            all_mu_values = []
            for slice_data in mu_slices:
                if slice_data is not None:
                    all_mu_values.extend(slice_data['mu'])
            all_mu_values = np.array(all_mu_values)
            mu_ref = (all_mu_values.min() + all_mu_values.max()) / 2.0
            print(f"Calculated mu_ref = {mu_ref:.2f} GPa (mean of global min/max)")
        else:
            print(f"Using provided mu_ref = {mu_ref:.2f} GPa")

        # Create anomaly slices
        anom_slices = []
        for slice_data in mu_slices:
            if slice_data is None:
                anom_slices.append(None)
            else:
                anom = (slice_data['mu'] - mu_ref) / mu_ref * 100  # Percentage
                anom_slices.append({
                    'lon': slice_data['lon'],
                    'lat': slice_data['lat'],
                    'anom': anom,
                    'depth_km': slice_data['depth_km']
                })

        return anom_slices, mu_ref

In [ ]:
# ============================================================================
# SECTION 5: Example usage
# ============================================================================

# Plot absolute mu slices
_plot_slice_panels_flex_surface(
    data_slices_scattered=mu_slice_scattered,
    # region=region, 
    depths_km=depth_levels,
    cmap="roma",
    # series=[5, 185, (185 - 5) / 20],  # Adjust to your data range
    # series=[5, 160, (160 - 5) / 20],  # Adjust to your data range
    series=[5, 160],  # Adjust to your data range
    # series=[5, 155, (155 - 5) / 15],  # Adjust to your data range
    # label="Shear modulus (GPa)",
    label="@~m@~ (GPa)",
    # show_colorbar_per_panel=True,  # NEW: enable per-panel colorbars
    filename=resultpath + "mu_horizontal_slices_surface.pdf",
    nrows=2,
    ncols=2,
    spacing="0.025d",  # ~3.3 km, matches mesh resolution
    tension=0.25,      # Moderate smoothness
    use_blockmedian=True, # Preprocess to remove warnings
    block_spacing="0.025d" # Same as final spacing
)

In [ ]:
# # Plot anomaly slices (using auto-calculated mu_ref)
# _plot_slice_panels_flex_surface(
#     data_slices_scattered=anom_slice_scattered,
#     # region=region_fault, 
#     depths_km=depth_levels,
#     cmap="roma",
#     series=[-100, 100, 100/10],  # Adjust to your data range
#     label="Anomaly (%)",
#     filename=resultpath + "mu_anomaly_horizontal_slices_surface.pdf",
#     nrows=2,
#     ncols=2,
#     spacing="0.025d",  # ~2.7 km, matches mesh resolution
#     tension=0.25,      # Moderate smoothness
#     use_blockmedian=True, # Preprocess to remove warnings
#     block_spacing="0.025d"  # Same as final spacing
# )

# OR: Use custom mu_ref
# # Option 1: Auto-calculated global mu_ref (mean of min/max) - ORIGINAL
# anom_slices_global, mu_ref_global = calculate_anomaly_slices(mu_slice_scattered)

# # Option 2: Custom mu_ref - ORIGINAL
# anom_slices_custom, _ = calculate_anomaly_slices(mu_slice_scattered, mu_ref=40.0)

# Option 3: Each slice's own mean - NEW!
anom_slices_local, mu_ref_list = calculate_anomaly_slices(mu_slice_scattered, use_slice_mean=True)

# ============================================================================
# Display anomaly range for each slice to inform colorbar selection
# ============================================================================

print("\n" + "="*60)
print("ANOMALY STATISTICS FOR EACH SLICE")
print("="*60)

global_anom_min = np.inf
global_anom_max = -np.inf

for i, anom_slice in enumerate(anom_slices_local):
    if anom_slice is None:
        print(f"Slice {i}: No data")
        continue

    anom_values = anom_slice['anom']
    depth = anom_slice['depth_km']

    slice_min = np.min(anom_values)
    slice_max = np.max(anom_values)
    slice_mean = np.mean(anom_values)
    slice_std = np.std(anom_values)

    # Update global range
    global_anom_min = min(global_anom_min, slice_min)
    global_anom_max = max(global_anom_max, slice_max)

    print(f"Depth {depth:6.1f} km: "
        f"min={slice_min:7.2f}%, max={slice_max:7.2f}%, "
        f"mean={slice_mean:7.2f}%, std={slice_std:6.2f}%")

    # If mu_ref was stored in the slice
    if 'mu_ref' in anom_slice:
        print(f"                (mu_ref = {anom_slice['mu_ref']:.2f} GPa)")

print("="*60)
print(f"GLOBAL RANGE: [{global_anom_min:.2f}%, {global_anom_max:.2f}%]")

# Suggest colorbar range (symmetric)
suggested_vmax = max(abs(global_anom_min), abs(global_anom_max))
# Round up to nearest 10 or 5
if suggested_vmax <= 50:
    suggested_vmax = np.ceil(suggested_vmax / 5) * 5
else:
    suggested_vmax = np.ceil(suggested_vmax / 10) * 10

print(f"SUGGESTED SYMMETRIC RANGE: ±{suggested_vmax:.1f}%")
print(f"SUGGESTED SERIES: [{-suggested_vmax:.1f}, {suggested_vmax:.1f}, {suggested_vmax/10:.1f}]")
print("="*60 + "\n")

# Create individual colorbar labels for each slice (showing its mu_ref)
colorbar_labels = []
for anom_slice in anom_slices_local:
    if anom_slice is None:
        colorbar_labels.append(None)
    else:
        if 'mu_ref' in anom_slice:
            label = f"Anomaly (%) rel. to {anom_slice['mu_ref']:.1f} GPa"
        else:
            label = "Anomaly (%)"
        colorbar_labels.append(label)

In [ ]:
## plot the anomaly
_plot_slice_panels_flex_surface(
    data_slices_scattered=anom_slices_local,
    depths_km=depth_levels,
    cmap="roma",
    series=[-suggested_vmax, suggested_vmax, suggested_vmax/10],  # Use calculated range
    # series=[-200, 200, 200/10],
    colorbar_labels=colorbar_labels,  # Pass individual labels
    show_colorbar_per_panel=True,     # Enable individual colorbars
    filename=resultpath + "mu_anomaly_horizontal_slices_surface.pdf",
    nrows=2,
    ncols=2,
    spacing="0.025d",  # ~2.7 km, matches mesh resolution
    tension=0.25,      # Moderate smoothness
    use_blockmedian=True, # Preprocess to remove warnings
    block_spacing="0.025d"  # Same as final spacing
    )

print("\n=== COMPARISON: Old vs New approach ===")
print("OLD: Mesh -> PyVista slice -> griddata (5.6 km) -> xyz2grd -> plot")
print("     Problem: Double interpolation, potential artifacts")
print("")
print("NEW: Mesh -> PyVista slice -> blockmedian -> pygmt.surface (3.3 km) -> plot")
print("     Benefits: Single smooth interpolation, respects mesh resolution, no warnings")
print("")
print("Anomaly calculation:")
print("  OLD: Relative to interpolated homogeneous model")
print("  NEW: Relative to mu_ref (flexible, can be changed)")
print("")
print("Parameters:")
print(f"  block_spacing='0.03d' ≈ 3.3 km (averages nearby points)")
print(f"  spacing='0.03d' ≈ 3.3 km (final grid resolution)")
print(f"  tension=0.25 (balanced smoothness)")
print("")
print("The blockmedian step:")
print("  - Eliminates duplicate/near-duplicate points")
print("  - Averages points within each grid cell")
print("  - Removes GMT surface warnings")
print("  - Still maintains ~3 km resolution matching your mesh")


In [ ]:
def _plot_slice_panels(data_slices, depths_km, cmap, series, label, filename):
    """Plot 3x3 panel of horizontal slices in PyGMT style"""

    fig = pygmt.Figure()
    with fig.subplot(nrows=3, ncols=3, figsize=("21c", "24c"), autolabel=False,
                    sharex=True, sharey=True, margins="0.35c/0.5c", frame=["WSne"]):
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
                    MAP_TITLE_OFFSET="0.3c", FONT_ANNOT_PRIMARY="7p")

        for idx_panel, (depth_km, data_grid) in enumerate(zip(depths_km, data_slices)):
            row = idx_panel // 3
            col = idx_panel % 3

            with fig.set_panel(panel=[row, col]):
                fig.basemap(region=region_fault, projection="M?",
                            frame=["a1f0.2"])  # Removed: f"+tDepth = {abs(depth_km):.0f} km"

                # Create grid from data
                # grid = pygmt.xyz2grd(x=lon_grid.ravel(), y=lat_grid.ravel(),
                #                     z=data_grid.ravel(), region=region_fault,
                #                     spacing=(lon_spacing, lat_spacing))
                
                grid = pygmt.surface(x=lon_grid.ravel(), y=lat_grid.ravel(),
                                    z=data_grid.ravel(), region=region_fault,
                                    spacing=(lon_spacing, lat_spacing),
                                    tension=0.25)  # 0=minimum curvature, 1=harmonic
                
                # Make colormap and plot
                pygmt.makecpt(cmap=cmap, series=series, background=True)
                fig.grdimage(grid=grid, cmap=True)

                fig.coast(shorelines="0.5p,black", area_thresh=100)

                fig.grdcontour(grid=plate_grd, levels=5, limit="-100/-5",
                            annotation="20+f6p", pen="0.4p,darkgray")
                # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray")

                # Add depth label as text inside plot at top-left corner
                fig.text(x=region_fault[0] + 0.05, y=region_fault[3] - 0.05,
                        text=f"Depth = {abs(depth_km):.0f} km",
                        font="10p,Helvetica-Bold,white", justify="TL",
                        fill="black@30", offset="0.1c/0.1c", pen="0.5p,white")

                # Colorbar (only on right column)
                # if idx_panel == len(depths_km)-1:
                if col == 2:
                    with pygmt.config(FONT_ANNOT_PRIMARY="9p"):
                        fig.colorbar(frame=["af", f"x+l{label}"])

                # ONLY ADDITION: Plot vertical slice locations on first panel
                if idx_panel == 0:
                    # Along-strike slices (constant x, blue dashed)
                    for x_km in x_strike_km:
                        x_m = x_km * 1e3
                        y_line = np.linspace(y_min * 1e3, y_max * 1e3, 5)
                        x_line = np.full_like(y_line, x_m)
                        lon_line, lat_line = utp.ckm2LLd(x_line + x0, y_line + y0, lon0, lat0, -rot)
                        fig.plot(x=lon_line, y=lat_line, pen="0.8p,white,--")

                    # Along-dip slices (constant y, red dashed)
                    for y_km in y_dip_km:
                        y_m = y_km * 1e3
                        x_line = np.linspace(x_min * 1e3, x_max * 1e3, 5)
                        y_line = np.full_like(x_line, y_m)
                        lon_line, lat_line = utp.ckm2LLd(x_line + x0, y_line + y0, lon0, lat0, -rot)
                        # print(lon_line, lat_line)
                        fig.plot(x=lon_line, y=lat_line, pen="0.8p,white,-.")

    fig.show()
    
    if flag_savefig:
        fig.savefig(filename, dpi=300)
        print(f"Saved slice figure to: {filename}")

    # return fig

In [ ]:
def _plot_slice_panels_flex(data_slices, depths_km, cmap, series, label, 
filename, 
                        nrows=3, ncols=3):
    """Plot horizontal slices in PyGMT style with flexible grid layout
    
    Parameters
    ----------
    data_slices : list
        List of 2D arrays containing data for each depth slice
    depths_km : list
        Depths in km for each slice
    cmap : str
        PyGMT colormap name
    series : list
        [min, max, step] for colormap
    label : str
        Label for colorbar
    filename : str
        Output filename
    nrows : int
        Number of rows in subplot grid (default 3)
    ncols : int
        Number of columns in subplot grid (default 3)
    """
    # Calculate figure size based on grid dimensions
    # Maintain aspect ratio: each panel ~7cm x 8cm
    fig_width = f"{ncols * 7}c"
    fig_height = f"{nrows * 8}c"

    fig = pygmt.Figure()
    with fig.subplot(nrows=nrows, ncols=ncols, figsize=(fig_width,
fig_height),
                    autolabel=False, sharex=True, sharey=True,
                    margins="0.35c/0.5c", frame=["WSne"]):
        pygmt.config(MAP_FRAME_TYPE="plain",
FONT_TITLE="10p,Helvetica-Bold",
                    MAP_TITLE_OFFSET="0.3c", FONT_ANNOT_PRIMARY="7p")

        for idx_panel, (depth_km, data_grid) in enumerate(zip(depths_km,
data_slices)):
            row = idx_panel // ncols
            col = idx_panel % ncols

            with fig.set_panel(panel=[row, col]):
                fig.basemap(region=region_fault, projection="M?",
                            frame=["a1f0.2"])

                # Create grid from data
                # grid = pygmt.xyz2grd(x=lon_grid.ravel(),y=lat_grid.ravel(),
                #                     z=data_grid.ravel(),region=region_fault,
                #                     spacing=(lon_spacing, lat_spacing))

                # grid = pygmt.surface(x=lon_grid.ravel(), y=lat_grid.ravel(),
                #                     z=data_grid.ravel(), region=region_fault,
                #                     spacing=(lon_spacing, lat_spacing),
                #                     tension=0.25)  # 0=minimum curvature, 1=harmonic

                # First block-average to actual resolution
                filtered = pygmt.blockmedian(x=lon_grid.ravel(), y=lat_grid.ravel(),
                                            z=data_grid.ravel(), region=region_fault,
                                            spacing="0.05d")  # Coarser spacing

                # Then use surface for smooth interpolation
                grid = pygmt.surface(data=filtered, region=region_fault,
                                    spacing=(lon_spacing, lat_spacing))

                # Make colormap and plot
                pygmt.makecpt(cmap=cmap, series=series, background=True)
                fig.grdimage(grid=grid, cmap=True)

                fig.coast(shorelines="0.5p,black", area_thresh=100)

                fig.grdcontour(grid=plate_grd, levels=5, limit="-100/-5",
                            annotation="20+f6p", pen="0.4p,darkgray")

                # Add depth label as text inside plot at top-left corner
                fig.text(x=region_fault[0] + 0.05, y=region_fault[3] -
0.05,
                        text=f"Depth = {abs(depth_km):.0f} km",
                        font="10p,Helvetica-Bold,white", justify="TL",
                        fill="black@30", offset="0.1c/0.1c",
pen="0.5p,white")

                # Colorbar (only on right column)
                if col == ncols - 1:  # Changed from col == 2
                    with pygmt.config(FONT_ANNOT_PRIMARY="9p"):
                        fig.colorbar(frame=["af", f"x+l{label}"])

                # ONLY ADDITION: Plot vertical slice locations on first panel
                if idx_panel == 0:
                    # Along-strike slices (constant x, blue dashed)
                    for x_km in x_strike_km:
                        x_m = x_km * 1e3
                        y_line = np.linspace(y_min * 1e3, y_max * 1e3, 5)
                        x_line = np.full_like(y_line, x_m)
                        lon_line, lat_line = utp.ckm2LLd(x_line + x0,
y_line + y0,
                                                        lon0, lat0,
-rot)
                        fig.plot(x=lon_line, y=lat_line,
pen="0.8p,white,--")

                    # Along-dip slices (constant y, red dashed)
                    for y_km in y_dip_km:
                        y_m = y_km * 1e3
                        x_line = np.linspace(x_min * 1e3, x_max * 1e3, 5)
                        y_line = np.full_like(x_line, y_m)
                        lon_line, lat_line = utp.ckm2LLd(x_line + x0,
y_line + y0,
                                                        lon0, lat0,
-rot)
                        fig.plot(x=lon_line, y=lat_line,
pen="0.8p,white,-.")

    fig.show()
    if flag_savefig:
        fig.savefig(filename, dpi=300)
        print(f"Saved slice figure to: {filename}")

In [ ]:
# # Horizontal slices of μ and μ anomaly (3x3 depth panels).
# depth_levels = np.arange(0.0, -80.1, -10.0)
# lon_spacing = 0.01
# lat_spacing = 0.01
# lon_vals = np.arange(region_fault[0], region_fault[1] + lon_spacing, lon_spacing)
# lat_vals = np.arange(region_fault[2], region_fault[3] + lat_spacing, lat_spacing)
# lon_grid, lat_grid = np.meshgrid(lon_vals, lat_vals)
# grid_shape = (len(lat_vals), len(lon_vals))

# from scipy.interpolate import griddata
# x_rot, y_rot = utp.LL2ckmd(lon_grid.ravel(), lat_grid.ravel(), lon0, lat0, rot)
# x_local = x_rot - x0
# y_local = y_rot - y0

# mu_points = mu_3d_grid.points
# mu_values_all = mu_3d_grid.point_data['shear modulus']
# mu_hom_points = mu_hom_grid.points
# mu_hom_values_all = mu_hom_grid.point_data['shear modulus']

# mu_slice_list = []
# anom_slice_list = []

# for depth_km in depth_levels:
#     z_value = depth_km * 1e3
#     sample_points = np.column_stack([
#         x_local,
#         y_local,
#         np.full_like(x_local, z_value)
#     ])

#     mu_values = griddata(mu_points, mu_values_all, sample_points, method='linear')
#     mu_hom_values = griddata(mu_hom_points, mu_hom_values_all, sample_points, method='linear')

#     mu_grid = mu_values.reshape(grid_shape)
#     mu_hom_grid_slice = mu_hom_values.reshape(grid_shape)

#     # mu_ref = mu_hom_grid_slice
#     # mu_ref = mu_3d_grid_mean

#     mu_anom_grid = (mu_grid - mu_ref) / mu_ref * 100.0

#     mu_slice_list.append(mu_grid)
#     anom_slice_list.append(mu_anom_grid)

# mu_slice_min = float(np.nanmin([np.nanmin(slc) for slc in mu_slice_list]))
# mu_slice_max = float(np.nanmax([np.nanmax(slc) for slc in mu_slice_list]))
# mu_slice_step = max((mu_slice_max - mu_slice_min) / 20.0, 1e-3)

# anom_slice_vmax = float(np.nanmax([np.nanmax(np.abs(slc)) for slc in anom_slice_list]))
# anom_slice_step = max(anom_slice_vmax / 10.0, 1e-3)

# print(f"\nμ range: [{mu_slice_min:.1f}, {mu_slice_max:.1f}] GPa, step: {mu_slice_step:.4f} GPa")
# print(f"Anomaly range: [{-anom_slice_vmax:.1f}, {anom_slice_vmax:.1f}] %, step: {anom_slice_step:.4f} %")

# # Define vertical slice positions
# x_strike_km = np.arange(-50, 51, 20)  # Along-strike slices (constant x)
# y_min, y_max = -100, 100  # km

# y_dip_km = np.arange(-50, 51, 20)     # Along-dip slices (constant y)
# x_min, x_max = -80, 120   # km


# mu_slice_file = resultpath + 'mu_horizontal_slices.png'
# _plot_slice_panels(mu_slice_list, depth_levels, cmap="rainbow",
#                    series=[mu_slice_min, mu_slice_max, mu_slice_step],
#                    label="μ (GPa)", filename=mu_slice_file)

# anom_slice_file = resultpath + 'mu_anomaly_horizontal_slices.png'
# _plot_slice_panels(anom_slice_list, depth_levels, cmap="roma",
#                    series=[-anom_slice_vmax, anom_slice_vmax, anom_slice_step],
#                    label="μ anomaly (%)", filename=anom_slice_file)



In [ ]:
# # Along-dip slices (constant y, red dashed)
# for y_km in y_dip_km:
#     y_m = y_km * 1e3
#     x_line = np.linspace(x_min * 1e3, x_max * 1e3, 5)
#     y_line = np.full_like(x_line, y_m)
#     lon_line, lat_line = utp.ckm2LLd(x_line, y_line, lon0, lat0, -rot)
#     print(lon_line, lat_line)

## 10.6. Vertical Slices Along Strike and Dip

Visualize μ structure in vertical cross-sections to understand how structure varies with depth along different directions.

**Slice Orientations:**
- **Along-strike slices**: Constant x (dip direction), varying y (strike) and z (depth)
- **Along-dip slices**: Constant y (strike direction), varying x (dip) and z (depth)

**Key Features:**
- Cartesian projection (x-z or y-z) in PyGMT
- Plate interface (fault) overlaid as reference line
- Shows where slip occurs relative to μ structure
- Multiple slice positions to capture spatial variability

**Implementation:**
- Use PyVista slicing along vertical planes
- Interpolate plate interface onto slice coordinates
- Plot in PyGMT with simple X-axis (distance) vs Z-axis (depth)

Visualize μ structure on vertical cross-sections along strike and dip directions,
with plate interface overlay for geological context.


In [ ]:
# PYGMT VERSION: Vertical along-dip slices using PyVista + PyGMT surface
# This approach: PyVista slice -> blockmedian -> pygmt.surface -> smooth grid

import numpy as np
import pygmt
import pyvista as pv
from scipy.interpolate import griddata

# ============================================================================
# SECTION 1: Create vertical slices along dip (constant y)
# ============================================================================

# Define y positions for along-dip slices
# y_dip_km = np.arange(-30, 31, 20)  # [-30, -10, 10, 30] - 4 slices
y_dip_km = np.arange(-10, 51, 20)  # [-30, -10, 10, 30] - 4 slices
y_dip_values = [y * 1e3 for y in y_dip_km]  # Convert to meters

# Define x-z region for slices
x_min, x_max = -80, 60  # km
z_min, z_max = -60, 0    # km
region_xz = [x_min, x_max, z_min, z_max]

# Choose what to plot: 'mu' or 'anomaly'
# plot_type = 'anomaly'  # Options: 'mu' or 'anomaly'
plot_type = 'mu'  # Options: 'mu' or 'anomaly'

# # Reference mu for anomaly calculations
# mu_ref = 88.5  # GPa (adjust as needed)
mu_ref = 80.85  # GPa (adjust as needed)

# Storage for scattered point data
mu_dip_slice_scattered = []

print("Creating vertical along-dip slices using PyVista slicing...")
for i, (y_km, y_val) in enumerate(zip(y_dip_km, y_dip_values)):
    # Extract slice from 3D grid
    normal = [0., 1., 0.]  # Normal in y direction
    origin = [0., y_val, 0.]

    slice_3d = mu_3d_grid.slice(normal=normal, origin=origin)

    if len(slice_3d.points) == 0:
        print(f"  y = {y_km} km: No data")
        mu_dip_slice_scattered.append(None)
        continue

    # Get sliced points and values (KEEP AS SCATTERED DATA)
    slice_points = slice_3d.points
    slice_mu = slice_3d['shear modulus']

    # Extract x and z coordinates (in meters)
    x_pts = slice_points[:, 0]  # Already in mesh coordinates
    z_pts = slice_points[:, 2]

    # Convert to km for PyGMT
    x_km = x_pts / 1e3
    z_km = z_pts / 1e3

    # Filter points to only keep those within region_xz
    mask = (
        (x_km >= x_min) & (x_km <= x_max) &
        (z_km >= z_min) & (z_km <= z_max)
    )

    x_km_filtered = x_km[mask]
    z_km_filtered = z_km[mask]
    mu_filtered = slice_mu[mask]

    if len(x_km_filtered) == 0:
        print(f"  y = {y_km} km: No data within region_xz")
        mu_dip_slice_scattered.append(None)
        continue

    # Display statistics for this slice
    mu_min = mu_filtered.min()
    mu_max = mu_filtered.max()
    mu_mean = np.mean(mu_filtered)
    mu_std = np.std(mu_filtered)

    print(f"  y = {y_km} km: {len(x_km)} total → {len(x_km_filtered)} within region_xz")
    print(f"    μ range: [{mu_min:.2f}, {mu_max:.2f}] GPa, "
        f"mean: {mu_mean:.2f} GPa, std: {mu_std:.2f} GPa")

    # Store as scattered points
    mu_dip_slice_scattered.append({
        'x': x_km_filtered,
        'z': z_km_filtered,
        'mu': mu_filtered,
        'y_km': y_km
    })

    print(f"  y = {y_km} km: {len(x_km)} scattered points")

print(f"\nCreated {len([s for s in mu_dip_slice_scattered if s is not None])} slices")

# ============================================================================
# SECTION 2: Calculate anomaly slices
# ============================================================================

# Collect all mu values to get global min/max
all_mu_values = []
for slice_data in mu_dip_slice_scattered:
    if slice_data is not None:
        all_mu_values.extend(slice_data['mu'])

all_mu_values = np.array(all_mu_values)
mu_global_min = all_mu_values.min()
mu_global_max = all_mu_values.max()

print(f"\nGlobal mu statistics:")
print(f"  Min: {mu_global_min:.2f} GPa")
print(f"  Max: {mu_global_max:.2f} GPa")

# Calculate reference mu for anomaly
# mu_ref = (mu_global_min + mu_global_max) / 2.0
print(f"  mu_ref for anomaly: {mu_ref:.2f} GPa")

# Create anomaly slices relative to mu_ref
anom_dip_slice_scattered = []
for slice_data in mu_dip_slice_scattered:
    if slice_data is None:
        anom_dip_slice_scattered.append(None)
    else:
        anom = (slice_data['mu'] - mu_ref) / mu_ref * 100  # Percentage
        anom_dip_slice_scattered.append({
            'x': slice_data['x'],
            'z': slice_data['z'],
            'anom': anom,
            'y_km': slice_data['y_km']
        })

print(f"Created anomaly slices relative to mu_ref = {mu_ref:.2f} GPa")

# ============================================================================
# SECTION 3: Prepare plate interface for each y location
# ============================================================================

# Prepare plate interface in ROTATED mesh coordinates
df_plate_for_slicing = df_plate.copy()
x_rot, y_rot = utp.LL2ckmd(df_plate['lon'].values, df_plate['lat'].values,
                           lon0, lat0, rot)
df_plate_for_slicing['x_mesh'] = x_rot - x0
df_plate_for_slicing['y_mesh'] = y_rot - y0
df_plate_for_slicing['z_mesh'] = df_plate['z'].values * 1e3  # Convert km to meters

plate_xy = np.column_stack([
    df_plate_for_slicing['x_mesh'].values,
    df_plate_for_slicing['y_mesh'].values
])
plate_z = df_plate_for_slicing['z_mesh'].values

# Interpolate plate interface at each y location
plate_interfaces = []
for y_val in y_dip_values:
    x_interface = np.linspace(x_min * 1e3, x_max * 1e3, 200)
    y_interface = np.full_like(x_interface, y_val)

    z_interface = griddata(
        plate_xy,
        plate_z,
        np.column_stack([x_interface, y_interface]),
        method='linear'
    )

    # Remove NaN values
    valid = ~np.isnan(z_interface)
    if valid.sum() > 0:
        interface_x_km = x_interface[valid] / 1e3
        interface_z_km = z_interface[valid] / 1e3
        plate_interfaces.append({
            'x': interface_x_km,
            'z': interface_z_km
        })
    else:
        plate_interfaces.append(None)

# ============================================================================
# SECTION 4: Plotting function for vertical slices using PyGMT
# ============================================================================

def _plot_vertical_slices_pygmt(data_slices_scattered, y_positions, cmap, series,
                                label, filename, plate_interfaces=None,
                                depth_levels=None, nrows=2, ncols=2,
                                spacing_x=2.0, spacing_z=2.0, tension=0.25,
                                use_blockmedian=True):
    """Plot vertical along-dip slices in PyGMT style using surface interpolation

    Parameters
    ----------
    data_slices_scattered : list of dict
        List of dicts with keys: 'x', 'z', and data key (e.g., 'mu', 'anom')
    y_positions : list
        Y positions in km for each slice
    cmap : str
        PyGMT colormap name
    series : list
        [min, max, step] for colormap
    label : str
        Label for colorbar
    filename : str
        Output filename
    plate_interfaces : list of dict, optional
        List of plate interface lines {'x': array, 'z': array}
    depth_levels : array-like, optional
        Depth levels to mark with horizontal lines (km)
    nrows : int
        Number of rows in subplot grid (default 2)
    ncols : int
        Number of columns in subplot grid (default 2)
    spacing_x : float
        Grid spacing in x direction (km)
    spacing_z : float
        Grid spacing in z direction (km)
    tension : float
        Tension parameter for surface (0=smooth, 1=tight to data, default 0.25)
    use_blockmedian : bool
        Whether to preprocess with blockmedian (default True)
    """
    # Determine data key (mu, anom, etc.)
    data_key = None
    for slice_data in data_slices_scattered:
        if slice_data is not None:
            for key in slice_data.keys():
                if key not in ['x', 'z', 'y_km']:
                    data_key = key
                    break
            break

    if data_key is None:
        raise ValueError("Could not determine data key in scattered slices")

    # Calculate figure size to maintain equal aspect ratio
    # For equal aspect: panel_width/panel_height = x_range/z_range
    x_range = region_xz[1] - region_xz[0]  # km (e.g., 60 - (-80) = 140 km)
    z_range = region_xz[3] - region_xz[2]  # km (e.g., 0 - (-60) = 60 km)
    aspect_ratio = x_range / z_range      # (e.g., 140/60 ≈ 2.33)

    # Set panel height and calculate width to maintain equal aspect (1 km = 1 km on plot)
    panel_height = 4.0  # cm
    panel_width = panel_height * aspect_ratio  # cm (e.g., 4 * 2.33 ≈ 9.3 cm)

    # Margins between panels
    margin_x = 0.15  # cm (horizontal spacing)
    margin_y = 0.6  # cm (vertical spacing)

    # Calculate total figure size including margins
    fig_width = f"{ncols * panel_width + (ncols-1) * margin_x}c"
    fig_height = f"{nrows * panel_height + (nrows-1) * margin_y}c"

    panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

    fig = pygmt.Figure()
    with fig.subplot(nrows=nrows, ncols=ncols, figsize=(fig_width, fig_height),
                    autolabel=False, sharex=True, sharey=True, frame=["WSne"],
                    margins=f"{margin_x}c/{margin_y}c",
                    ):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
                    MAP_TITLE_OFFSET="0.3c", FONT_ANNOT_PRIMARY="8p")

        for idx_panel, (y_km, slice_data) in enumerate(zip(y_positions, data_slices_scattered)):
            row = idx_panel // ncols
            col = idx_panel % ncols

            with fig.set_panel(panel=[row, col]):
                # Basemap with Cartesian projection (equal aspect ratio)
                # projection="X?" automatically maintains aspect ratio based on region
                # Frame annotation: "xa20f10" = annotate every 20 km, frame ticks every 10 km
                # Use FONT_LABEL config for consistent font size
                with pygmt.config(FONT_LABEL="9p"):
                    if row == nrows - 1:
                        # Bottom row: show x-axis label
                        fig.basemap(region=region_xz, projection="X?",
                                    frame=["xa20f10+lAlong-dip (km)", "ya20f10"])   #, "WSne"
                    else:
                        # Other rows: no x-axis label
                        fig.basemap(region=region_xz, projection="X?",
                                    frame=["xa20f10", "ya20f10"])   #, "WSne"

                # Check if we have data for this slice
                if slice_data is None:
                    fig.text(x=(region_xz[0] + region_xz[1])/2,
                            y=(region_xz[2] + region_xz[3])/2,
                            text="No data", font="12p,Helvetica-Bold,red",
                            justify="CM")
                else:
                    print(f"  Panel {idx_panel} (y={y_km} km): Original points: {len(slice_data['x'])}", end='')

                    # Preprocess with blockmedian
                    if use_blockmedian:
                        filtered_data = pygmt.blockmedian(
                            x=slice_data['x'],
                            y=slice_data['z'],
                            z=slice_data[data_key],
                            region=region_xz,
                            spacing=f"{spacing_x}/{spacing_z}",
                            registration="gridline"
                        )
                        x_filtered = filtered_data.iloc[:, 0].values
                        z_filtered = filtered_data.iloc[:, 1].values
                        data_filtered = filtered_data.iloc[:, 2].values
                        print(f" → Filtered to: {len(x_filtered)} points")
                    else:
                        x_filtered = slice_data['x']
                        z_filtered = slice_data['z']
                        data_filtered = slice_data[data_key]

                    # Use pygmt.surface on filtered points
                    grid = pygmt.surface(
                        x=x_filtered,
                        y=z_filtered,
                        z=data_filtered,
                        region=region_xz,
                        spacing=f"{spacing_x}/{spacing_z}",
                        tension=tension,
                        registration="gridline"
                    )

                    # Make colormap and plot
                    pygmt.makecpt(cmap=cmap, series=series, background=True)
                    fig.grdimage(grid=grid, cmap=True)  #, interpolation="n"

                    # Add depth level lines if provided
                    if depth_levels is not None:
                        for depth in depth_levels:
                            fig.plot(x=[region_xz[0], region_xz[1]],
                                   y=[depth, depth],
                                   pen="0.5p,white,--")

                    # Add plate interface if provided
                    if plate_interfaces is not None and idx_panel < len(plate_interfaces):
                        interface = plate_interfaces[idx_panel]
                        if interface is not None:
                            fig.plot(x=interface['x'], y=interface['z'],
                                   pen="1.5p,black")

                    # Colorbar (adaptive to layout)
                    # Calculate colorbar width as ~80% of panel width for consistency
                    cbar_width = panel_width * 0.8
                    cbar_height = 0.2  # cm

                    # if ncols == 1:
                    #     # For single column layout: horizontal colorbar on bottom panel only
                    #     if row == nrows - 1:
                    #         with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                    #             # fig.colorbar(frame=["af", f"x+l{label}"],
                    #             #            position=f"JBC+w{cbar_width:.1f}c/{cbar_height}c+h")
                    #             fig.colorbar(frame=[f"a{series[2]*4}f{series[2]}", f"x+l{label}"],
                    #                        position=f"JBC+w{cbar_width:.1f}c/{cbar_height}c+h")

                    # else:
                    #     # For multi-column layout: vertical colorbar on right column
                    #     # Use panel_height for vertical colorbar length
                    #     cbar_length = panel_height * 0.8
                    #     if col == ncols - 1:
                    #         with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                    #             # fig.colorbar(frame=["af", f"x+l{label}"],
                    #             #            position=f"JMR+w{cbar_length:.1f}c/{cbar_height}c+o0.5c/0c")
                    #             fig.colorbar(frame=[f"a{series[2]*4}f{series[2]}", f"x+l{label}"],
                    #                        position=f"JMR+w{cbar_length:.1f}c/{cbar_height}c+o0.5c/0c")

                    if row == 0:
                        with pygmt.config(FONT_ANNOT_PRIMARY="9p", FONT_LABEL="11p"):
                            # fig.colorbar(frame=["af", f"x+l{label}"], position="JBC+h+o0/0.25c")
                            fig.colorbar(frame=[f"a{series[2]*4}f{series[2]}", f"x+l{label}"],
                                    position=f"JBC+w{cbar_width:.1f}c/{cbar_height}c+h+o0/0.25c")
                            # fig.colorbar(frame=[f"a{30}f{15}", f"x+l{panel_label}"],
                            #         position="JBC+h+o0/0.25c")


                # Y-axis label (only on left column or middle row for single column)
                if ncols == 1:
                    # For single column layout: label on middle panel only
                    if row == nrows // 2:
                        with pygmt.config(FONT_LABEL="9p"):
                            fig.basemap(frame=["y+lDepth (km)"])
                else:
                    # For multi-column layout: label on left column
                    if col == 0:
                        with pygmt.config(FONT_LABEL="9p"):
                            fig.basemap(frame=["y+lDepth (km)"])

                # Add y label at top-right corner (matching horizontal slice style)
                fig.text(x=region_xz[1] - 2, y=region_xz[3] - 2,
                        text=f"Along-strike={y_km} km",
                        font="10p,Helvetica-Bold,white", justify="TR",
                        fill="gray60", offset="0.1c/0.1c")   # fill="black@50", pen="0.5p,white"

                #plot panel label
                panel_label = panel_labels[idx_panel]
                fig.text(text=panel_label, x=region_xz[0]+1, y=region_xz[2]+3, 
                            font="11p,Helvetica-Bold,black", justify="LB")

    fig.show()
    if flag_savefig:
        # fig.savefig(filename)   #, dpi=300, transparent=False
        fig.savefig(filename.replace('.pdf', '.png'), dpi=600)

    print(f"Saved slice figure to: {filename}")

# ============================================================================
# SECTION 5: Example usage
# ============================================================================

# Determine color limits
if plot_type == 'anomaly':
    data_slices = anom_dip_slice_scattered
    all_vals = np.concatenate([s['anom'][~np.isnan(s['anom'])]
                               for s in data_slices if s is not None])
    vmax = 100  # Symmetric for anomaly
    vmin = -vmax
    cmap = "roma"
    label = "μ Anomaly (%)"
    series = [vmin, vmax, (vmax - vmin) / 20.0]
else:  # mu
    data_slices = mu_dip_slice_scattered
    all_vals = np.concatenate([s['mu'][~np.isnan(s['mu'])]
                               for s in data_slices if s is not None])
    vmin = 5
    vmax = 160
    cmap = "roma"
    # label="@~m@~ (GPa)",
    label = "μ (GPa)"
    series = [vmin, vmax, (vmax - vmin) / 20.0]

print(f"\nColor range: [{vmin}, {vmax}]")

# Depth levels to mark
depth_levels = np.arange(-20.0, -50.1, -10.0)

# Plot vertical slices
_plot_vertical_slices_pygmt(
    data_slices_scattered=data_slices,
    y_positions=y_dip_km,
    cmap=cmap,
    series=series,
    label=label,
    filename=resultpath + f'mu_vertical_slices_dip_{plot_type}_pygmt.pdf',
    plate_interfaces=plate_interfaces,
    depth_levels=depth_levels,
    nrows=2,  # 4 rows (changed from 2)
    ncols=2,  # 1 column (changed from 2)
    spacing_x=2.5,   # 2 km in x direction
    spacing_z=2.5,   # 2 km in z direction
    tension=0.25,
    use_blockmedian=True
)

print("\n=== COMPARISON: Matplotlib vs PyGMT approach ===")
print("MATPLOTLIB: PyVista slice -> griddata (regular grid) -> contourf")
print("            Problem: Double interpolation, potential artifacts")
print("")
print("PYGMT:      PyVista slice -> blockmedian -> pygmt.surface -> grdimage")
print("            Benefits: Single smooth interpolation, professional quality")
print("")
print("Figure properties:")
print(f"  Region: x=[{x_min}, {x_max}] km, z=[{z_min}, {z_max}] km")
print(f"  Panel size: 4.0 cm height (width auto-adjusted for equal aspect)")
print(f"  Panel margins: 0.4 cm (horizontal) × 0.25 cm (vertical)")
print(f"  Axis labels: 9p font (consistent for both x and y)")
print(f"  Grid spacing: 2.5 km × 2.5 km")
print(f"  Tension: 0.25 (balanced smoothness)")


In [ ]:
# def extract_vertical_slice_pygmt(mu_grid, normal, origin, slice_axis='y-z',
#                                    field_name='shear modulus'):
#     """
#     Extract vertical slice from PyVista grid and prepare for PyGMT plotting.

#     Parameters:
#     -----------
#     mu_grid : pyvista.UnstructuredGrid
#         The 3D grid with mu values
#     normal : list
#         Normal vector for the slice plane [nx, ny, nz]
#     origin : list
#         Origin point for the slice plane [x, y, z]
#     slice_axis : str
#         Which axes to plot: 'y-z' for along-strike, 'x-z' for along-dip
#     field_name : str
#         Name of the field to extract from the grid

#     Returns:
#     --------
#     slice_df : pd.DataFrame
#         DataFrame with columns [coord1, z, mu] ready for PyGMT
#     """
#     # Slice the grid
#     slice_obj = mu_grid.slice(normal=normal, origin=origin)

#     if len(slice_obj.points) == 0:
#         print(f"Warning: Empty slice at origin {origin}")
#         return None

#     # Extract coordinates and mu values
#     points = slice_obj.points  # [N, 3] array
#     mu_vals = slice_obj[field_name]

#     # Select appropriate coordinates based on slice direction
#     if slice_axis == 'y-z':
#         # Along-strike slice (constant x)
#         coord1 = points[:, 1] / 1e3  # y in km
#         coord1_name = 'y_km'
#     elif slice_axis == 'x-z':
#         # Along-dip slice (constant y)
#         coord1 = points[:, 0] / 1e3  # x in km
#         coord1_name = 'x_km'
#     else:
#         raise ValueError(f"Unknown slice_axis: {slice_axis}")

#     z_km = points[:, 2] / 1e3  # z in km

#     # Create DataFrame
#     slice_df = pd.DataFrame({
#         coord1_name: coord1,
#         'z_km': z_km,
#         'mu': mu_vals
#     })

#     return slice_df


In [ ]:
# Ensure mu_3d_grid and mu_hom_grid are PyVista objects
# If they were overwritten, reload them

if not hasattr(mu_3d_grid, 'slice'):
    print("Reloading mu_3d_grid as PyVista object...")
    mu_3d_grid = load_xdmf_as_pyvista(mu_3d_file)

if not hasattr(mu_hom_grid, 'slice'):
    print("Reloading mu_hom_grid as PyVista object...")
    mu_hom_grid = load_xdmf_as_pyvista(mu_hom_file)

print(f"mu_3d_grid type: {type(mu_3d_grid)}")
print(f"mu_hom_grid type: {type(mu_hom_grid)}")
print("Ready for slicing!")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from scipy.interpolate import griddata

# Define x positions for along-strike slices: -50 to 50 km, increment 20 km
x_strike_km = np.arange(-50, 51, 20)  # [-50, -30, -10, 10, 30, 50] - 6 slices
x_strike_values = [x * 1e3 for x in x_strike_km]  # Convert to meters

# Choose what to plot: 'mu' or 'anomaly'
plot_type = 'mu'  # Options: 'mu' or 'anomaly'
# plot_type = 'anomaly'  # Options: 'mu' or 'anomaly'

# Prepare plate interface in ROTATED mesh coordinates
df_plate_for_slicing = df_plate.copy()
# Convert lon/lat back to rotated mesh coordinates
x_rot, y_rot = utp.LL2ckmd(df_plate['lon'].values, df_plate['lat'].values, lon0, lat0,
rot)
df_plate_for_slicing['x_mesh'] = x_rot - x0
df_plate_for_slicing['y_mesh'] = y_rot - y0
df_plate_for_slicing['z_mesh'] = df_plate['z'].values * 1e3  # Convert km to meters

print(f"Plate interface mesh coordinates:")
print(f"  X: [{df_plate_for_slicing['x_mesh'].min()/1e3:.1f}, \
{df_plate_for_slicing['x_mesh'].max()/1e3:.1f}] km")
print(f"  Y: [{df_plate_for_slicing['y_mesh'].min()/1e3:.1f}, \
{df_plate_for_slicing['y_mesh'].max()/1e3:.1f}] km")
print(f"  Z: [{df_plate_for_slicing['z_mesh'].min()/1e3:.1f}, \
{df_plate_for_slicing['z_mesh'].max()/1e3:.1f}] km")

# Prepare 2D interpolation for plate interface (x,y) -> z
plate_xy = np.column_stack([
    df_plate_for_slicing['x_mesh'].values,
    df_plate_for_slicing['y_mesh'].values
])
plate_z = df_plate_for_slicing['z_mesh'].values

# Create matplotlib figure with 3x2 subplots
fig, axes = plt.subplots(3, 2, figsize=(7, 5), dpi=300, sharex=True, sharey=True)
axes = axes.flatten()

# Define region for all panels (y-z space)
y_min, y_max = -100, 100  # km
z_min, z_max = -80, 0     # km

# Create regular grid in y-z space for consistent interpolation
n_y_points = 200
n_z_points = 160
y_grid = np.linspace(y_min * 1e3, y_max * 1e3, n_y_points)  # meters
z_grid = np.linspace(z_min * 1e3, z_max * 1e3, n_z_points)  # meters
yy, zz = np.meshgrid(y_grid, z_grid)
grid_shape = yy.shape

# Collect all values to determine global color limits
all_values = []

for i, (x_km, x_val) in enumerate(zip(x_strike_km, x_strike_values)):
    # Extract slice from both grids
    normal = [1., 0., 0.]  # Normal in x direction
    origin = [x_val, 0., 0.]

    slice_3d = mu_3d_grid.slice(normal=normal, origin=origin)
    slice_hom = mu_hom_grid.slice(normal=normal, origin=origin)

    if len(slice_3d.points) == 0:
        print(f"Warning: Empty slice at x = {x_km} km")
        continue

    # Interpolate BOTH grids to the same regular y-z grid using linear method
    mu_3d_interp = griddata(
        slice_3d.points[:, 1:3],  # y, z coordinates
        slice_3d['shear modulus'],
        (yy, zz),
        method='linear'
    )

    mu_hom_interp = griddata(
        slice_hom.points[:, 1:3],  # y, z coordinates
        slice_hom['shear modulus'],
        (yy, zz),
        method='linear'
    )

    # Choose what to plot
    if plot_type == 'anomaly':
        # Compute mu anomaly in percentage
        # reference mu for anomaly calculations
        # mu_ref = mu_hom_interp
        # mu_ref = mu_3d_grid_mean

        plot_values = (mu_3d_interp - mu_ref) / mu_ref * 100
        cmap = 'cmc.roma'
        cbar_label = 'μ Anomaly (%)'
    else:  # plot_type == 'mu'
        plot_values = mu_3d_interp
        cmap = 'gist_rainbow_r'
        cbar_label = 'Shear Modulus μ (GPa)'

    all_values.append(plot_values.ravel())

# Determine color limits
all_vals = np.concatenate([v[~np.isnan(v)] for v in all_values])
if plot_type == 'anomaly':
    # Symmetric range for diverging colormap
    vmax = np.abs(all_vals).max()
    if het3d_str == '_DeShon3D_ref_4':
        # vmax = (155 - mu_ref) / mu_ref * 100
        vmax = 150  # Cap at 150% for better color contrast
        vmax = 160  # Cap at 160% for better color contrast
    vmin = -vmax
    print(f"Anomaly range: [{all_vals.min():.1f}, {all_vals.max():.1f}]% (using symmetric±{vmax:.1f}%)")
else:
    vmin = all_vals.min()
    vmax = all_vals.max()
    if het3d_str == '_DeShon3D_ref_4':
        vmax = 155  # Cap at 155 for better color contrast
        vmax = 180  # Cap at 155 for better color contrast
    print(f"μ range: [{all_vals.min():.1f}, {all_vals.max():.1f}] GPa (using {vmax:.1f} GPa)")

# Now plot with determined vmin/vmax
for i, (x_km, x_val) in enumerate(zip(x_strike_km, x_strike_values)):
    ax = axes[i]

    # Extract slice from both grids
    normal = [1., 0., 0.]
    origin = [x_val, 0., 0.]

    slice_3d = mu_3d_grid.slice(normal=normal, origin=origin)
    slice_hom = mu_hom_grid.slice(normal=normal, origin=origin)

    if len(slice_3d.points) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=14)
        continue

    # Interpolate BOTH grids to the same regular y-z grid using linear method
    mu_3d_interp = griddata(
        slice_3d.points[:, 1:3],  # y, z coordinates
        slice_3d['shear modulus'],
        (yy, zz),
        method='linear'
    )

    mu_hom_interp = griddata(
        slice_hom.points[:, 1:3],  # y, z coordinates
        slice_hom['shear modulus'],
        (yy, zz),
        method='linear'
    )

    if plot_type == 'anomaly':
        # reference mu for anomaly calculations
        # mu_ref = mu_hom_interp
        # mu_ref = mu_3d_grid_mean

        plot_values = (mu_3d_interp - mu_ref) / mu_ref * 100
        cmap = 'cmc.roma'
    else:
        plot_values = mu_3d_interp
        cmap = 'gist_rainbow_r'

    # Convert grid to km for plotting
    yy_km = yy / 1e3
    zz_km = zz / 1e3

    # Plot with pcolormesh (works well with regular grids)
    contour_levels = np.linspace(vmin, vmax, 20)
    cs = ax.contourf(yy_km, zz_km, plot_values, levels=contour_levels, cmap=cmap,
                    vmin=vmin, vmax=vmax, extend='both')

    # Add horizontal lines for depth levels
    for depth in depth_levels:
        ax.axhline(y=depth, color='white', linestyle='--', linewidth=0.5, zorder=5)

    # Interpolate plate interface at this specific x location
    # Create evenly spaced y points
    y_interface = np.linspace(y_min * 1e3, y_max * 1e3, 200)  # 200 points
    x_interface = np.full_like(y_interface, x_val)  # All at same x

    # Interpolate z at these (x, y) positions
    z_interface = griddata(
        plate_xy,
        plate_z,
        np.column_stack([x_interface, y_interface]),
        method='linear'  # Linear interpolation for smooth result
    )

    # Remove NaN values (points outside convex hull)
    valid = ~np.isnan(z_interface)
    if valid.sum() > 0:
        interface_y_km = y_interface[valid] / 1e3
        interface_z_km = z_interface[valid] / 1e3

        # Plot with high zorder
        ax.plot(interface_y_km, interface_z_km, 'k-', linewidth=1.5, zorder=10)
        # ax.plot(interface_y_km, interface_z_km, 'w--', linewidth=1.5, zorder=11)

    # Set limits and labels
    ax.set_xlim(y_min, y_max)
    ax.set_ylim(z_min, z_max)
    ax.set_aspect('equal')  # Equal aspect ratio
    ax.set_title(f'x = {x_km} km', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.set_yticks(np.arange(-80, 1, 20))

    # Labels on left column and bottom row
    if i % 2 == 0:  # Left column (2 columns)
        ax.set_ylabel('Depth (km)', fontsize=11)
    if i >= 4:  # Bottom row (last 2 panels)
        ax.set_xlabel('Along-strike (km)', fontsize=11)

# Add colorbar at bottom (horizontal)
plt.tight_layout(rect=[0, 0.05, 1, 1])  # Leave space at bottom for colorbar
cbar_ax = fig.add_axes([0.15, 0.01, 0.7, 0.02])  # [left, bottom, width, height]
cbar = fig.colorbar(cs, cax=cbar_ax, orientation='horizontal')
cbar.set_label(cbar_label, fontsize=12)
if het3d_str == '_DeShon3D_ref_4':
    if plot_type == 'mu':
        cbar.set_ticks(np.arange(vmin, vmax+1, 20))  # 5, 25, 45, 65, 85, 105, 125, 145
    else:
        cbar.set_ticks(np.arange(-150, 150+1, 50))
elif het3d_str == '_mul55u25':
    if plot_type == 'mu':
        cbar.set_ticks(np.arange(25, 55+1, 5))  # 5, 25, 45, 65, 85, 105, 125, 145
    else:
        cbar.set_ticks(np.arange(-40, 40+1, 10))

plt.show()

if flag_savefig:
    output_file = resultpath + f'mu_vertical_slices_strike_{plot_type}.png'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\nAlong-strike slices saved to: {output_file}")

In [ ]:
# def plot_vertical_slice(ax, y_km, mesh_3d, mesh_hom, mu_ref, plate_xy, 
# plate_z, 
#                         x_min, x_max, z_min, z_max, plot_type='mu',
#                         cmap='rainbow', vmin=None, vmax=None):
#     """
#     Plot a single vertical slice at constant y position.
    
#     Parameters
#     ----------
#     ax : matplotlib axis
#         The axis to plot on
#     y_km : float
#         Y position in km for this slice
#     mesh_3d : pyvista mesh
#         The 3D heterogeneous mu mesh
#     mesh_hom : pyvista mesh
#         The homogeneous mu mesh (for reference)
#     mu_ref : float
#         Reference mu value for anomaly calculation
#     plate_xy : array
#     Plate interface x,y coordinates
#     plate_z : array
#         Plate interface z coordinates
#     x_min, x_max : float
#         X-axis limits in km
#     z_min, z_max : float
#         Z-axis limits in km
#     plot_type : str
#         'mu' or 'anomaly'
#     cmap : str
#         Colormap name
#     vmin, vmax : float
#         Color scale limits
    
#     Returns
#     -------
#     contourf : matplotlib contourf object
#         For colorbar creation
#     """
#     y_m = y_km * 1e3

#     # Create vertical slice through PyVista mesh
#     normal = [0, 1, 0]  # Normal vector for y-constant slice
#     origin = [0, y_m, 0]
#     slice_3d = mesh_3d.slice(normal=normal, origin=origin)
#     slice_hom = mesh_hom.slice(normal=normal, origin=origin)

#     if slice_3d.n_points == 0:
#         ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
#                 ha='center', va='center', fontsize=12)
#         return None

#     # Extract coordinates
#     x_slice = slice_3d.points[:, 0] / 1e3  # Convert to km
#     z_slice = slice_3d.points[:, 2] / 1e3

#     # Create regular grid for contour plotting
#     x_grid = np.linspace(x_min, x_max, 200)
#     z_grid = np.linspace(z_min, z_max, 200)
#     xx, zz = np.meshgrid(x_grid, z_grid)

#     # Interpolate to grid based on plot type
#     if plot_type == 'mu':
#         mu_3d_interp = griddata(
#             (slice_3d.points[:, 0], slice_3d.points[:, 2]),
#             slice_3d['shear modulus'],
#             (xx * 1e3, zz * 1e3),
#             method='linear'
#         )
#         plot_values = mu_3d_interp / 1e9  # Convert Pa to GPa
#     else:  # anomaly
#         mu_3d_interp = griddata(
#             (slice_3d.points[:, 0], slice_3d.points[:, 2]),
#             slice_3d['shear modulus'],
#             (xx * 1e3, zz * 1e3),
#             method='linear'
#         )
#         plot_values = (mu_3d_interp - mu_ref) / mu_ref * 100

#     # Plot filled contours
#     contourf = ax.contourf(xx, zz, plot_values, levels=50,
#                             cmap=cmap, vmin=vmin, vmax=vmax, extend='both')

#     # Interpolate plate interface at this y position
#     x_interface = np.linspace(x_min * 1e3, x_max * 1e3, 100)
#     y_interface = np.full_like(x_interface, y_m)

#     z_interface = griddata(
#         plate_xy, plate_z,
#         np.column_stack([x_interface, y_interface]),
#         method='linear'
#     )

#     # Plot plate interface
#     valid = ~np.isnan(z_interface)
#     if valid.sum() > 0:
#         interface_x_km = x_interface[valid] / 1e3
#         interface_z_km = z_interface[valid] / 1e3
#         ax.plot(interface_x_km, interface_z_km, 'k-', linewidth=1.5,
# zorder=10)

#     # Set limits and styling
#     ax.set_xlim(x_min, x_max)
#     ax.set_ylim(z_min, z_max)
#     ax.set_aspect('equal')
#     ax.grid(True, alpha=0.3, linewidth=0.5)
#     ax.set_yticks(np.arange(-80, 1, 20))

#     # Add text at top-right corner inside plot
#     ax.text(0.98, 0.98, f'y = {y_km} km',
#             transform=ax.transAxes, fontsize=10, fontweight='bold',
#             ha='right', va='top',
#             bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
#                     alpha=0.8, edgecolor='gray', linewidth=0.8))

#     return contourf


# # ===== MAIN PLOTTING CODE =====

# # Define y positions for along-dip slices
# y_dip_km = np.arange(-30, 31, 20)  # [-30, -10, 10, 30] - 4 slices

# # Choose what to plot: 'mu' or 'anomaly'
# plot_type = 'mu'  # Options: 'mu' or 'anomaly'

# # Prepare plate interface in ROTATED mesh coordinates
# df_plate_for_slicing = df_plate.copy()
# x_rot, y_rot = utp.LL2ckmd(df_plate['lon'].values, df_plate['lat'].values,

#                             lon0, lat0, rot)
# df_plate_for_slicing['x_mesh'] = x_rot - x0
# df_plate_for_slicing['y_mesh'] = y_rot - y0
# df_plate_for_slicing['z_mesh'] = df_plate['z'].values * 1e3  # Convert km to meters

# print(f"Plate interface mesh coordinates:")
# print(f"  X: [{df_plate_for_slicing['x_mesh'].min()/1e3:.1f}, "
#     f"{df_plate_for_slicing['x_mesh'].max()/1e3:.1f}] km")
# print(f"  Y: [{df_plate_for_slicing['y_mesh'].min()/1e3:.1f}, "
#     f"{df_plate_for_slicing['y_mesh'].max()/1e3:.1f}] km")
# print(f"  Z: [{df_plate_for_slicing['z_mesh'].min()/1e3:.1f}, "
#     f"{df_plate_for_slicing['z_mesh'].max()/1e3:.1f}] km")

# # Prepare 2D interpolation for plate interface
# plate_xy = np.column_stack([
#     df_plate_for_slicing['x_mesh'].values,
#     df_plate_for_slicing['y_mesh'].values
# ])
# plate_z = df_plate_for_slicing['z_mesh'].values

# # Define region limits
# x_min, x_max = -100, 100  # km
# z_min, z_max = -80, 0     # km

# # Set colormap and limits based on plot type
# if plot_type == 'mu':
#     cmap = 'rainbow'
#     vmin, vmax = 20, 60  # GPa (adjust as needed)
#     cbar_label = 'μ (GPa)'
# else:  # anomaly
#     cmap = 'RdBu_r'
#     vmin, vmax = -30, 30  # % (adjust as needed)
#     cbar_label = 'μ anomaly (%)'

# # ===== CREATE FIGURE WITH FLEXIBLE LAYOUT =====
# # Easy to change number of rows/columns based on number of slices
# n_slices = len(y_dip_km)
# ncols = 2
# nrows = (n_slices + ncols - 1) // ncols  # Ceiling division

# fig, axes = plt.subplots(nrows, ncols, figsize=(8, 4*nrows/2),
#                         dpi=300, sharex=True, sharey=True)
# axes = axes.flatten()

# # Make tick labels smaller for all axes
# for ax in axes:
#     ax.tick_params(axis='both', labelsize=8)

# # Adjust spacing between subplots
# plt.subplots_adjust(hspace=0.15, wspace=0.15)

# # Plot each slice
# for i, y_km in enumerate(y_dip_km):
#     contourf = plot_vertical_slice(
#         axes[i], y_km, mu_3d_grid, mu_hom_grid, mu_ref,  # Changed these parameters
#         plate_xy, plate_z,
#         x_min, x_max, z_min, z_max,
#         plot_type=plot_type, cmap=cmap, vmin=vmin, vmax=vmax
#     )

#     # Labels on left column and bottom row
#     if i % ncols == 0:  # Left column
#         axes[i].set_ylabel('Depth (km)', fontsize=11)
#     if i >= n_slices - ncols:  # Bottom row
#         axes[i].set_xlabel('Along-dip (km)', fontsize=11)

# # Hide extra subplots if any
# for i in range(n_slices, len(axes)):
#     axes[i].axis('off')

# # Add colorbar at bottom (horizontal)
# if contourf is not None:
#     cbar = plt.colorbar(contourf, ax=axes, orientation='horizontal',
#                         pad=0.08, aspect=30, shrink=0.8)
#     cbar.ax.tick_params(labelsize=8)  # Smaller colorbar tick labels
#     cbar.set_label(cbar_label, fontsize=10)

# plt.show()

# if flag_savefig:
#     plt.savefig(f'vertical_slices_{plot_type}.png', dpi=300, bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from scipy.interpolate import griddata

# Define y positions for along-dip slices: -50 to 50 km, increment 20 km
# y_dip_km = np.arange(-50, 51, 20)  # [-50, -30, -10, 10, 30, 50] - 6 slices
y_dip_km = np.arange(-30, 31, 20)  # [-30, -10, 10, 30] - 4 slices
y_dip_values = [y * 1e3 for y in y_dip_km]  # Convert to meters

# Choose what to plot: 'mu' or 'anomaly'
# plot_type = 'mu'  # Options: 'mu' or 'anomaly'
plot_type = 'anomaly'  # Options: 'mu' or 'anomaly'

# Prepare plate interface in ROTATED mesh coordinates
df_plate_for_slicing = df_plate.copy()
# Convert lon/lat back to rotated mesh coordinates
x_rot, y_rot = utp.LL2ckmd(df_plate['lon'].values, df_plate['lat'].values, lon0, lat0,
rot)
df_plate_for_slicing['x_mesh'] = x_rot - x0
df_plate_for_slicing['y_mesh'] = y_rot - y0
df_plate_for_slicing['z_mesh'] = df_plate['z'].values * 1e3  # Convert km to meters

print(f"Plate interface mesh coordinates:")
print(f"  X: [{df_plate_for_slicing['x_mesh'].min()/1e3:.1f}, \
{df_plate_for_slicing['x_mesh'].max()/1e3:.1f}] km")
print(f"  Y: [{df_plate_for_slicing['y_mesh'].min()/1e3:.1f}, \
{df_plate_for_slicing['y_mesh'].max()/1e3:.1f}] km")
print(f"  Z: [{df_plate_for_slicing['z_mesh'].min()/1e3:.1f}, \
{df_plate_for_slicing['z_mesh'].max()/1e3:.1f}] km")

# Prepare 2D interpolation for plate interface (x,y) -> z
plate_xy = np.column_stack([
    df_plate_for_slicing['x_mesh'].values,
    df_plate_for_slicing['y_mesh'].values
])
plate_z = df_plate_for_slicing['z_mesh'].values

# Create matplotlib figure with 3x2 subplots
# fig, axes = plt.subplots(3, 2, figsize=(7, 5), dpi=300, sharex=True, sharey=True)
fig, axes = plt.subplots(2, 2, figsize=(8, 4), dpi=300, sharex=True, sharey=True)
axes = axes.flatten()

# Make tick labels smaller for all axes
for ax in axes:
    ax.tick_params(axis='both', labelsize=8)  # Change 8 to your desired size

# Define region for all panels (x-z space)
# x_min, x_max = -80, 120  # km
x_min, x_max = -100, 100  # km
z_min, z_max = -80, 0    # km

mu_3d_grid_mean = mu_3d_grid['shear modulus'].mean()
print(mu_3d_grid['shear modulus'].mean())

depth_levels = np.arange(-20.0, -50.1, -10.0)  # 0 to -80 km, 4 depths

# Create regular grid in x-z space for consistent interpolation
n_x_points = 200
n_z_points = 160
x_grid = np.linspace(x_min * 1e3, x_max * 1e3, n_x_points)  # meters
z_grid = np.linspace(z_min * 1e3, z_max * 1e3, n_z_points)  # meters
xx, zz = np.meshgrid(x_grid, z_grid)
grid_shape = xx.shape

# Collect all values to determine global color limits
all_values = []

for i, (y_km, y_val) in enumerate(zip(y_dip_km, y_dip_values)):
    # Extract slice from both grids
    normal = [0., 1., 0.]  # Normal in y direction
    origin = [0., y_val, 0.]

    slice_3d = mu_3d_grid.slice(normal=normal, origin=origin)
    slice_hom = mu_hom_grid.slice(normal=normal, origin=origin)

    if len(slice_3d.points) == 0:
        print(f"Warning: Empty slice at y = {y_km} km")
        continue

    # Interpolate BOTH grids to the same regular x-z grid using linear method
    mu_3d_interp = griddata(
        slice_3d.points[:, [0, 2]],  # x, z coordinates
        slice_3d['shear modulus'],
        (xx, zz),
        method='linear'
    )

    mu_hom_interp = griddata(
        slice_hom.points[:, [0, 2]],  # x, z coordinates
        slice_hom['shear modulus'],
        (xx, zz),
        method='linear'
    )

    # Choose what to plot
    if plot_type == 'anomaly':
        # Compute mu anomaly in percentage
        # reference mu for anomaly calculations
        # mu_ref = mu_hom_interp
        # mu_ref = mu_3d_grid_mean
        mu_ref = 88.5

        plot_values = (mu_3d_interp - mu_ref) / mu_ref * 100
        cmap = 'cmc.roma'
        cbar_label = 'μ Anomaly (%)'
    else:  # plot_type == 'mu'
        plot_values = mu_3d_interp
        cmap = 'gist_rainbow_r'
        cbar_label = 'Shear Modulus μ (GPa)'

    all_values.append(plot_values.ravel())

# Determine color limits
all_vals = np.concatenate([v[~np.isnan(v)] for v in all_values])
if plot_type == 'anomaly':
    # Symmetric range for diverging colormap
    vmax = np.abs(all_vals).max()
    if het3d_str == '_DeShon3D_ref_4':
        # vmax = (155 - mu_ref) / mu_ref * 100
        vmax = 150  # Cap at 150% for better color contrast
        vmax = 160  # Cap at 160% for better color contrast
        vmax = 100  # Cap at 160% for better color contrast
    vmin = -vmax
    print(f"Anomaly range: [{all_vals.min():.1f}, {all_vals.max():.1f}]% (using symmetric±{vmax:.1f}%)")
else:
    vmin = all_vals.min()
    vmax = all_vals.max()
    if het3d_str == '_DeShon3D_ref_4':
        vmax = 155  # Cap at 155 for better color contrast
        vmax = 180  # Cap at 155 for better color contrast
    print(f"μ range: [{all_vals.min():.1f}, {all_vals.max():.1f}] GPa (using {vmax:.1f} GPa)")

# Now plot with determined vmin/vmax
for i, (y_km, y_val) in enumerate(zip(y_dip_km, y_dip_values)):
    ax = axes[i]

    # Extract slice from both grids
    normal = [0., 1., 0.]
    origin = [0., y_val, 0.]

    slice_3d = mu_3d_grid.slice(normal=normal, origin=origin)
    slice_hom = mu_hom_grid.slice(normal=normal, origin=origin)

    if len(slice_3d.points) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=14)
        continue

    # Interpolate BOTH grids to the same regular x-z grid using linear method
    mu_3d_interp = griddata(
        slice_3d.points[:, [0, 2]],  # x, z coordinates
        slice_3d['shear modulus'],
        (xx, zz),
        method='linear'
    )

    mu_hom_interp = griddata(
        slice_hom.points[:, [0, 2]],  # x, z coordinates
        slice_hom['shear modulus'],
        (xx, zz),
        method='linear'
    )

    if plot_type == 'anomaly':
        # reference mu for anomaly calculations
        # mu_ref = mu_hom_interp
        # mu_ref = mu_3d_grid_mean

        plot_values = (mu_3d_interp - mu_ref) / mu_ref * 100
        cmap = 'cmc.roma'
    else:
        plot_values = mu_3d_interp
        cmap = 'gist_rainbow_r'

    # Convert grid to km for plotting
    xx_km = xx / 1e3
    zz_km = zz / 1e3

    # Plot with contourf (works well with regular grids)
    contour_levels = np.linspace(vmin, vmax, 20)
    cs = ax.contourf(xx_km, zz_km, plot_values, levels=contour_levels, cmap=cmap,
                    vmin=vmin, vmax=vmax, extend='both')

    # Add horizontal lines for depth levels
    for depth in depth_levels:
        ax.axhline(y=depth, color='white', linestyle='--', linewidth=0.5, zorder=5)

    # Interpolate plate interface at this specific y location
    # Create evenly spaced x points
    x_interface = np.linspace(x_min * 1e3, x_max * 1e3, 200)  # 200 points
    y_interface = np.full_like(x_interface, y_val)  # All at same y

    # Interpolate z at these (x, y) positions
    z_interface = griddata(
        plate_xy,
        plate_z,
        np.column_stack([x_interface, y_interface]),
        method='linear'  # Linear interpolation for smooth result
    )

    # Remove NaN values (points outside convex hull)
    valid = ~np.isnan(z_interface)
    if valid.sum() > 0:
        interface_x_km = x_interface[valid] / 1e3
        interface_z_km = z_interface[valid] / 1e3

        # Plot with high zorder
        ax.plot(interface_x_km, interface_z_km, 'k-', linewidth=1.5, zorder=10)
        # ax.plot(interface_x_km, interface_z_km, 'w--', linewidth=1.5, zorder=11)

    # Set limits and labels
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(z_min, z_max)
    ax.set_aspect('equal')  # Equal aspect ratio
    # ax.set_title(f'y = {y_km} km', fontsize=12, fontweight='bold')
    # Add text at top-right corner inside plot
    ax.text(0.98, 0.98, f'y = {y_km} km', transform=ax.transAxes, fontsize=8, fontweight='bold',
            ha='right', va='top', )
    ax.grid(True, alpha=0.3, linewidth=0.5)
    ax.set_yticks(np.arange(-80, 1, 20))

    # Labels on left column and bottom row
    if i % 2 == 0:  # Left column (2 columns)
        ax.set_ylabel('Depth (km)', fontsize=8)
    if i >= 2:  # Bottom row (last 2 panels)
        ax.set_xlabel('Along-dip (km)', fontsize=8)

# Add colorbar at bottom (horizontal)
plt.tight_layout(rect=[0, 0.05, 1, 1])  # Leave space at bottom for colorbar
cbar_ax = fig.add_axes([0.15, 0.01, 0.7, 0.02])  # [left, bottom, width, height]
cbar = fig.colorbar(cs, cax=cbar_ax, orientation='horizontal')
cbar.set_label(cbar_label, fontsize=8)
cbar.ax.tick_params(labelsize=8)  # Make colorbar tick labels smaller
if het3d_str == '_DeShon3D_ref_4':
    if plot_type == 'mu':
        cbar.set_ticks(np.arange(vmin, vmax+1, 20))  # 5, 25, 45, 65, 85, 105, 125, 145
    else:
        # cbar.set_ticks(np.arange(-150, 150+1, 50))
        cbar.set_ticks(np.arange(-100, 100+1, 25))

elif het3d_str == '_mul55u25':
    if plot_type == 'mu':
        cbar.set_ticks(np.arange(25, 55+1, 5))  # 5, 25, 45, 65, 85, 105, 125, 145
    else:
        cbar.set_ticks(np.arange(-40, 40+1, 10))

plt.subplots_adjust(hspace=0.0, wspace=0.1)

plt.show()

if flag_savefig:
    output_file = resultpath + f'mu_vertical_slices_dip_{plot_type}.png'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\nAlong-dip slices saved to: {output_file}")

In [ ]:
mu_ref

In [ ]:
# # If you prefer PyGMT style, use denser interpolation
# # This creates a regular grid from scattered slice points

# from scipy.interpolate import griddata

# x_strike_km = [-60, -30, 0, 30, 60]
# x_strike_values = [x * 1e3 for x in x_strike_km]

# fig = pygmt.Figure()

# y_min, y_max = -120, 120
# z_min, z_max = -80, 5
# region = [y_min, y_max, z_min, z_max]

# # Create dense regular grid for interpolation
# y_grid = np.linspace(y_min, y_max, 400)  # Increased from 200
# z_grid = np.linspace(z_min, z_max, 300)  # Increased from 150
# yy, zz = np.meshgrid(y_grid, z_grid)

# for i, (x_km, x_val) in enumerate(zip(x_strike_km, x_strike_values)):
#     normal = [1., 0., 0.]
#     origin = [x_val, 0., 0.]

#     slice_mesh = mu_3d_grid.slice(normal=normal, origin=origin)

#     if len(slice_mesh.points) == 0:
#         continue

#     # Extract slice data
#     y_slice = slice_mesh.points[:, 1] / 1e3
#     z_slice = slice_mesh.points[:, 2] / 1e3
#     mu_slice = slice_mesh['shear modulus']

#     # Interpolate to DENSE regular grid
#     mu_grid = griddata((y_slice, z_slice), mu_slice, (yy, zz),
#                        method='linear')  # Use linear for smoother result

#     if i > 0:
#         fig.shift_origin(xshift='12c')

#     fig.basemap(region=region, projection="X10c/8c",
#                 frame=['WSne', 'xaf+l"Along-strike (km)"', 'yaf+l"Depth (km)"'])

#     # Create DataFrame for xyz2grd
#     y_flat = yy.ravel()
#     z_flat = zz.ravel()
#     mu_flat = mu_grid.ravel()

#     # Remove NaN values
#     valid = ~np.isnan(mu_flat)

#     # Create grid with MUCH smaller spacing
#     grid_3d = pygmt.xyz2grd(x=y_flat[valid], y=z_flat[valid], z=mu_flat[valid],
#                             region=region, spacing=[0.5, 0.3])  # Smaller spacing

#     fig.grdimage(grid=grid_3d, cmap='viridis', nan_transparent=True)

#     # Overlay plate interface
#     tolerance = 5e3
#     interface_mask = np.abs(df_plate_interface['x_local'] - x_val) < tolerance

#     if interface_mask.sum() > 0:
#         interface_y_km = df_plate_interface.loc[interface_mask, 'y_local'].values / 1e3
#         interface_z_km = df_plate_interface.loc[interface_mask, 'z_local'].values / 1e3
#         sort_idx = np.argsort(interface_y_km)
#         fig.plot(x=interface_y_km[sort_idx], y=interface_z_km[sort_idx], pen="2p,white,-")

#     fig.text(text=f"x = {x_km} km", position="TC", offset="0c/0.3c",
#              font="12p,Helvetica-Bold")

#     if i == len(x_strike_km) - 1:
#         fig.colorbar(position="JMR+o1c/0c+w8c/0.4c", frame=[f'af+l"@~m@~ (GPa)"'])

# fig.show()

# if flag_savefig:
#     output_file = resultpath + 'mu_vertical_slices_strike_pygmt_dense.png'
#     fig.savefig(output_file, dpi=300)
#     print(f"\nDense PyGMT slices saved to: {output_file}")


## 11. Correlation: Δμ vs. Slip Bias

Now correlate the μ contrast with slip bias to see the structure-slip trade-off.

**For locking problems:** We use the **s_dip component** (downdip coupling) because:
- Interseismic locking is primarily downdip motion (plate convergence direction)
- s_dip represents coupling ratio after normalization by amp
- Strike-slip component is typically small in subduction locking
- The physics of interest is how much the fault is locked in the convergence direction

In [ ]:
# # Interpolate delta_mu to slip node locations
# from scipy.interpolate import griddata

# # Remove NaN values from delta_mu
# valid_mask = ~np.isnan(delta_mu)
# x_fault_valid = fault_coords[valid_mask, 0]  # Use fault_coords, not facet_centers
# y_fault_valid = fault_coords[valid_mask, 1]
# z_fault_valid = fault_coords[valid_mask, 2]
# delta_mu_valid = delta_mu[valid_mask]

# # Interpolate to slip node locations (use m_s_3d which has slip_diff)
# delta_mu_at_slip = griddata(
#     (x_fault_valid, y_fault_valid, z_fault_valid),
#     delta_mu_valid,
#     (m_s_3d['x'].values, m_s_3d['y'].values, m_s_3d['z'].values),
#     method='nearest'  # Use nearest neighbor for robustness
# )

# print(f"Interpolated Δμ to {len(delta_mu_at_slip)} slip nodes")
# print(f"Range: [{np.nanmin(delta_mu_at_slip):.2f}, {np.nanmax(delta_mu_at_slip):.2f}] GPa")

In [ ]:
# # Correlation plot: Δμ vs. slip difference
# fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# # Determine labels based on problem type
# if problem_type == 'locking':
#     slip_diff_label = 'Coupling Difference (3D - Hom)'
#     y_axis_label = 'Δ Coupling (3D - Hom)'
# else:
#     slip_diff_label = 'Slip Difference (3D - Hom)'
#     y_axis_label = 'Δ Slip (m)'

# # Panel 1: Scatter plot
# ax1 = axes[0]
# scatter = ax1.scatter(delta_mu_at_slip, m_s_3d['slip_diff'],
#                      c=m_s_3d['depth'], s=20, cmap='viridis', alpha=0.6)
# ax1.set_xlabel('Δμ = μ_HW - μ_FW (GPa)', fontsize=12)
# ax1.set_ylabel(y_axis_label, fontsize=12)
# ax1.set_title(f'Correlation: μ Contrast vs. {slip_diff_label}', fontsize=13, fontweight='bold')
# ax1.grid(True, alpha=0.3)
# ax1.axhline(0, color='k', linestyle='--', linewidth=0.5)
# ax1.axvline(0, color='k', linestyle='--', linewidth=0.5)

# cbar = plt.colorbar(scatter, ax=ax1)
# cbar.set_label('Depth (km)')

# # Compute correlation
# valid_corr = ~np.isnan(delta_mu_at_slip)
# if np.sum(valid_corr) > 0:
#     from scipy.stats import pearsonr
#     corr_coef, p_value = pearsonr(delta_mu_at_slip[valid_corr],
#                                    m_s_3d['slip_diff'].values[valid_corr])
#     ax1.text(0.05, 0.95, f'Correlation: r = {corr_coef:.3f}\np-value = {p_value:.2e}',
#             transform=ax1.transAxes, fontsize=10, va='top',
#             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# # Panel 2: Binned statistics
# ax2 = axes[1]
# n_bins = 15
# bins = np.linspace(np.nanmin(delta_mu_at_slip), np.nanmax(delta_mu_at_slip), n_bins)
# bin_centers = (bins[:-1] + bins[1:]) / 2
# bin_means = []
# bin_stds = []

# for i in range(len(bins)-1):
#     mask = (delta_mu_at_slip >= bins[i]) & (delta_mu_at_slip < bins[i+1])
#     if np.sum(mask) > 0:
#         bin_means.append(np.mean(m_s_3d['slip_diff'].values[mask]))
#         bin_stds.append(np.std(m_s_3d['slip_diff'].values[mask]))
#     else:
#         bin_means.append(np.nan)
#         bin_stds.append(np.nan)

# bin_means = np.array(bin_means)
# bin_stds = np.array(bin_stds)

# ax2.errorbar(bin_centers, bin_means, yerr=bin_stds, fmt='o-', capsize=5,
#             color='blue', ecolor='lightblue', linewidth=2, markersize=6)
# ax2.set_xlabel('Δμ = μ_HW - μ_FW (GPa)', fontsize=12)
# ax2.set_ylabel(f'Mean {y_axis_label}', fontsize=12)
# ax2.set_title('Binned Average', fontsize=13, fontweight='bold')
# ax2.grid(True, alpha=0.3)
# ax2.axhline(0, color='k', linestyle='--', linewidth=0.5)
# ax2.axvline(0, color='r', linestyle='--', linewidth=0.5, alpha=0.5)

# plt.tight_layout()
# if flag_savefig:
#     plt.savefig(resultpath + 'correlation_delta_mu_slip_bias.png', dpi=300, bbox_inches='tight')
# plt.show()

# print(f"\nExpected behavior (slip_diff = 3D - hom):")
# if problem_type == 'locking':
#     print(f"  - Negative Δμ (HW softer): Positive slip_diff → hom underestimates coupling")
#     print(f"  - Positive Δμ (HW stiffer): Negative slip_diff → hom overestimates coupling")
# else:
#     print(f"  - Negative Δμ (HW softer): Positive slip_diff → hom underestimates slip")
#     print(f"  - Positive Δμ (HW stiffer): Negative slip_diff → hom overestimates slip")

## 12. Final Summary Figure

Create the complete 5-panel figure for publication

In [ ]:
# # Final comprehensive figure
# fig = plt.figure(figsize=(18, 12))
# gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)

# # Determine labels based on problem type
# if problem_type == 'locking':
#     slip_diff_label_short = 'Δ Coupling'
#     slip_diff_label_long = 'Coupling Difference (3D - Hom)'
# else:
#     slip_diff_label_short = 'Δ Slip'
#     slip_diff_label_long = 'Slip Difference (3D - Hom)'

# # Panel 1: μ anomaly (relative to homogeneous)
# ax1 = fig.add_subplot(gs[0, 0])
# x_km = fault_coords[:, 0] / 1e3
# y_km = fault_coords[:, 1] / 1e3
# vmax_anom = abs(mu_anomaly).max()
# sc1 = ax1.scatter(x_km, y_km, c=mu_anomaly, s=5, cmap='RdBu_r',
#                  vmin=-vmax_anom, vmax=vmax_anom, alpha=0.8)
# ax1.set_xlabel('X (km)')
# ax1.set_ylabel('Y (km)')
# ax1.set_title('(a) μ Anomaly on Fault', fontsize=12, fontweight='bold')
# ax1.set_aspect('equal')
# ax1.grid(True, alpha=0.3)
# cbar1 = plt.colorbar(sc1, ax=ax1)
# cbar1.set_label('(μ₃ᴅ - μₕₒₘ)/μₕₒₘ (%)')

# # Panel 2: Δμ across fault (NOW USES SAME fault_coords as panel 1!)
# ax2 = fig.add_subplot(gs[0, 1])
# # Use fault_coords consistently, not facet_centers
# vmax_delta = np.nanmax(np.abs(delta_mu))
# sc2 = ax2.scatter(x_km, y_km, c=delta_mu, s=50, cmap='RdBu_r',
#                  vmin=-vmax_delta, vmax=vmax_delta, alpha=0.8)
# ax2.set_xlabel('X (km)')
# ax2.set_ylabel('Y (km)')
# ax2.set_title('(b) Δμ Across Fault (HW - FW)', fontsize=12, fontweight='bold')
# ax2.set_aspect('equal')
# ax2.grid(True, alpha=0.3)
# cbar2 = plt.colorbar(sc2, ax=ax2)
# cbar2.set_label('Δμ (GPa)')

# # Panel 3: Slip difference (s_dip for locking, slip_mag for coseismic)
# # Now using m_s_3d which has slip_diff
# ax3 = fig.add_subplot(gs[0, 2])
# slip_x_km = m_s_3d['x'].values / 1e3
# slip_y_km = m_s_3d['y'].values / 1e3
# vmax_slip = abs(m_s_3d['slip_diff']).max()
# sc3 = ax3.scatter(slip_x_km, slip_y_km, c=m_s_3d['slip_diff'], s=50,
#                  cmap='RdBu_r', vmin=-vmax_slip, vmax=vmax_slip, alpha=0.8)
# ax3.set_xlabel('X (km)')
# ax3.set_ylabel('Y (km)')
# ax3.set_title(f'(c) {slip_diff_label_long}', fontsize=12, fontweight='bold')
# ax3.set_aspect('equal')
# ax3.grid(True, alpha=0.3)
# cbar3 = plt.colorbar(sc3, ax=ax3)
# cbar3.set_label(slip_diff_label_short)

# # Panel 4: Correlation scatter
# ax4 = fig.add_subplot(gs[1, :2])
# scatter = ax4.scatter(delta_mu_at_slip, m_s_3d['slip_diff'],
#                      c=m_s_3d['depth'], s=20, cmap='viridis', alpha=0.6)
# ax4.set_xlabel('Δμ = μ_HW - μ_FW (GPa)', fontsize=12)
# ax4.set_ylabel(slip_diff_label_short, fontsize=12)
# ax4.set_title('(d) Structure-Slip Trade-off', fontsize=12, fontweight='bold')
# ax4.grid(True, alpha=0.3)
# ax4.axhline(0, color='k', linestyle='--', linewidth=0.5)
# ax4.axvline(0, color='k', linestyle='--', linewidth=0.5)
# cbar4 = plt.colorbar(scatter, ax=ax4)
# cbar4.set_label('Depth (km)')

# if np.sum(valid_corr) > 0:
#     ax4.text(0.05, 0.95, f'r = {corr_coef:.3f}, p = {p_value:.2e}',
#             transform=ax4.transAxes, fontsize=10, va='top',
#             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# # Panel 5: Binned statistics
# ax5 = fig.add_subplot(gs[1, 2])
# ax5.errorbar(bin_centers, bin_means, yerr=bin_stds, fmt='o-', capsize=5,
#             color='blue', ecolor='lightblue', linewidth=2, markersize=6)
# ax5.set_xlabel('Δμ (GPa)', fontsize=11)
# ax5.set_ylabel(f'Mean {slip_diff_label_short}', fontsize=11)
# ax5.set_title('(e) Binned Average', fontsize=12, fontweight='bold')
# ax5.grid(True, alpha=0.3)
# ax5.axhline(0, color='k', linestyle='--', linewidth=0.5)
# ax5.axvline(0, color='r', linestyle='--', linewidth=0.5, alpha=0.5)

# plt.show()

# if flag_savefig:
#     plt.savefig(resultpath + 'complete_mu_slip_analysis.png', dpi=300, bbox_inches='tight')

# print(f"\nComplete analysis figure saved to: {resultpath}complete_mu_slip_analysis.png")

## Summary

This notebook successfully implemented:
- ✅ Loaded mesh and extracted fault vertices (3165 nodes)
- ✅ Extracted μ from XDMF files at fault locations
- ✅ Computed μ anomaly on fault surface
- ✅ **Computed Δμ across fault using layer-based averaging (μ_HW - μ_FW)**
- ✅ Loaded slip data (3D inv vs. homogeneous inv)
- ✅ **Normalized slip data appropriately for locking problems (s_dip / amp)**
- ✅ **Computed slip difference as 3D - hom (stored in m_s_3d)**
- ✅ **Correlated Δμ with coupling bias (s_dip component)**
- ✅ Created comprehensive visualization

### Key Methodological Improvements:

**Layer-based μ sampling:**
- Samples μ over a thin layer (e.g., 10 km thick) on each side of fault
- Averages multiple sampling points within each layer (e.g., 5 points)
- More robust than single-point sampling - less sensitive to local anomalies
- Better represents effective μ structure relevant to Green's functions

### Key Findings:

1. **μ Contrast Pattern:**
   - Mean Δμ shows whether overriding plate is systematically stiffer/softer
   - Spatial variations in Δμ correlate with fault structure
   - Layer-averaged values provide smoother, more representative estimates

2. **Structure-Slip Trade-off (Locking Problem):**
   - Correlation coefficient quantifies the relationship between Δμ and coupling bias
   - **Slip difference = 3D - hom** (positive means hom underestimates)
   - Where HW is softer (Δμ < 0): slip_diff > 0 → hom underestimates coupling
   - Where HW is stiffer (Δμ > 0): slip_diff < 0 → hom overestimates coupling
   - **Focus on s_dip component** (downdip coupling) which represents fault locking

3. **Practical Implications:**
   - Homogeneous inversions introduce systematic bias in coupling estimates
   - Bias direction depends on whether HW is softer or stiffer than FW
   - Bias magnitude correlates with structure contrast across the fault
   - 3D structure essential for accurate locking/coupling estimation in interseismic inversions
   - Misestimating μ structure leads to misinterpreting fault locking patterns
   - Layer-based averaging provides more stable Δμ estimates for correlation analysis